### Using DTW to match precipitaiton events to the consequent recharge event. 

In [1]:
"""Pull recharge events for 10 random NE wells using DTW-based matching."""

import numpy as np
import pandas as pd
import geopandas as gpd
import os
from dtw import dtw
import matplotlib.pyplot as plt
import io
import requests
import json
import time



Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



#### Loading in the data. Code pulled from Groundwater_x_Precipitation_FIXED.ipynb

In [2]:
well_sites = r"C:\Users\romin\OneDrive\Groundwater\RpSy Data\Site information for all selected wells.xlsx"
state_boundaries = r"C:\Users\romin\OneDrive\Groundwater\cb_2022_us_state_500k"
recharge_dir = r"C:/Users/romin/OneDrive/Groundwater/RpSy Data"
daymet_dir = "./daymet"

NE_states = [
    "Connecticut", "Maine", "Massachusetts", "New Hampshire",
    "Rhode Island", "Vermont", "New Jersey", "New York", "Pennsylvania",
]

# ============================================================
# Load wells and filter to NE states
# ============================================================

def load_sites(path=well_sites):
    df = pd.read_excel(path)
    df = df.rename(columns={
        "ID": "usgs_id",
        "Lat": "lat",
        "Long": "lon",
        "depth (m)": "depth",
    })
    return df

def select_ne_wells(df_sites, states_shp=state_boundaries):
    states = gpd.read_file(states_shp)
    ne = states[states["NAME"].isin(NE_states)].set_crs(epsg=4326, allow_override=True)
    gdf_sites = gpd.GeoDataFrame(
        df_sites,
        geometry=gpd.points_from_xy(df_sites["lon"], df_sites["lat"]),
        crs="EPSG:4326",
    )
    joined = gpd.sjoin(gdf_sites, ne[["NAME", "geometry"]], how="inner", predicate="within")
    joined = joined.rename(columns={"NAME": "state"}).drop(columns=["index_right"])
    return joined

df_sites = load_sites()
df_ne_wells = select_ne_wells(df_sites)


#### Determining the start and end date of RpSy data.

In [3]:
def get_start_end_dates(df, date_col="Date"):
    df[date_col] = pd.to_datetime(df[date_col])
    return df[date_col].min(), df[date_col].max() 

start_dates = []
end_dates = []
for wid in df_ne_wells["usgs_id"]:
    df = pd.read_csv(f"{recharge_dir}/{wid}.csv")
    start_date, end_date = get_start_end_dates(df)
    start_dates.append(start_date)
    end_dates.append(end_date)

df_ne_wells["start_date"] = start_dates
df_ne_wells["end_date"] = end_dates
df_ne_wells["record_length"] = (df_ne_wells["end_date"] - df_ne_wells["start_date"]).dt.days / 365.25

#### Filtering for wells with a record length >= 5 years.

In [4]:
eligible_wells = df_ne_wells[df_ne_wells["record_length"] >= 5].copy()
print(f"Wells with >=5 year records: {len(eligible_wells)}")

selected_wells = eligible_wells.sample(n=10, random_state=42)
print(selected_wells[["usgs_id", "record_length", "start_date", "end_date"]])

Wells with >=5 year records: 163
             usgs_id  record_length start_date   end_date
426  425803077151201      18.995209 2003-10-02 2022-09-30
403  421746074180201      15.994524 2006-10-02 2022-09-30
422  424520070562401      13.993155 1985-10-02 1999-09-30
306  404639074230001      12.993840 2009-10-02 2022-09-30
356  414330076280501      23.994524 1998-10-02 2022-09-30
274  400229075104601       9.993155 2012-10-02 2022-09-30
458  444904074455201      19.994524 2002-10-02 2022-09-30
302  404140077354001      16.996578 1999-10-02 2016-09-30
364  415228070554601       7.994524 2000-10-02 2008-09-30
439  434217073010601       5.993155 2016-10-02 2022-09-30


#### Functions to assist the DTW package and the calculated Precipitation/Recharge event table. Written by Professor David Litwin. 

In [5]:
# DTW helper functions
def get_events(values, threshold):
    """Return start/end index pairs for runs of consecutive values > threshold."""
    above = (values > threshold).astype(int)
    padded = np.concatenate(([0], above, [0]))
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0] - 1
    return starts, ends

def precip_recharge_event_table(df, precip_col, recharge_col, alignment, precip_threshold=0.0):
    """Return a table of precipitation and recharge events based on DTW alignment."""
    dates = df.index
    precip_vals = df[precip_col].values
    recharge_vals = df[recharge_col].values
    starts, ends = get_events(precip_vals, precip_threshold)

    rows = []
    prev_recharge_idx = None
    for s, e in zip(starts, ends):
        precip_idx = np.arange(s, e + 1)
        mask = np.isin(alignment.index2, precip_idx)
        recharge_idx = np.unique(alignment.index1[mask])

        if prev_recharge_idx is not None and recharge_idx.size > 0:
            test = recharge_idx > prev_recharge_idx.max()
            recharge_idx = recharge_idx[test]   # dropped silently, no print

        row = {
            "precip_start_date": dates[s],
            "precip_end_date": dates[e],
            "precip_total": precip_vals[precip_idx].sum(),
            "precip_peak": precip_vals[precip_idx].max(),
        }
        if recharge_idx.size == 0:
            row.update({
                "recharge_start_date": pd.NaT,
                "recharge_end_date": pd.NaT,
                "recharge_total": np.nan,
                "recharge_peak": np.nan,
                "n_recharge_days": 0,
            })
        else:
            row.update({
                "recharge_start_date": dates[recharge_idx.min()],
                "recharge_end_date": dates[recharge_idx.max()],
                "recharge_total": recharge_vals[recharge_idx].sum(),
                "recharge_peak": recharge_vals[recharge_idx].max(),
                "n_recharge_days": recharge_idx.size,
            })
        rows.append(row)
        prev_recharge_idx = recharge_idx if recharge_idx.size > 0 else prev_recharge_idx

    return pd.DataFrame(rows)

def check_overlapping_recharge_events(event_table):
    """Check for overlapping recharge events in the event table."""
    sorted_table = event_table.sort_values("recharge_start_date")
    overlaps = []
    for i in range(len(sorted_table) - 1):
        current_end = sorted_table.iloc[i]["recharge_end_date"]
        next_start = sorted_table.iloc[i + 1]["recharge_start_date"]
        if pd.notna(current_end) and pd.notna(next_start) and current_end >= next_start:
            overlap_days = (current_end - next_start).days + 1
            overlaps.append((i, current_end, next_start, overlap_days))
    return overlaps

#### Running the DTW function for the 163 eligible wells. 

In [6]:
os.makedirs("dtw_event_tables", exist_ok=True)

eligible_wells = df_ne_wells[df_ne_wells["record_length"] >= 5].copy()
print(f"Total wells with >=5 year records: {len(eligible_wells)}")

def run_dtw_pipeline(site_id, recharge_dir=recharge_dir, daymet_dir=daymet_dir,
                      out_dir="dtw_event_tables",
                      precip_threshold=2.5, min_lag=0, max_lag=5):
    recharge_path = f"{recharge_dir}/{site_id}.csv"
    precip_path = f"{daymet_dir}/{site_id}.csv"

    if not os.path.exists(recharge_path) or not os.path.exists(precip_path):
        return None, "missing file"

    df_recharge = pd.read_csv(recharge_path)
    df_precip = pd.read_csv(precip_path)

    df_precip["Date"] = pd.to_datetime(df_precip['year'] * 1000 + df_precip['yday'], format='%Y%j')
    df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])
    df_recharge.set_index("Date", inplace=True)
    df_precip.set_index("Date", inplace=True)

    df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
    df.drop(columns=["year", "yday"], inplace=True, errors="ignore")

    if len(df) < 30:
        return None, "too short"

    recharge = df["RpSy (m)"].values
    precip = df["prcp (mm/day)"].values

    if np.std(recharge) == 0 or np.std(precip) == 0:
        return None, "zero variance"

    recharge_norm = recharge / np.std(recharge)
    precip_norm = precip / np.std(precip)

    def causal_window(iw, jw, query_size, reference_size, min_lag=min_lag, max_lag=max_lag, **kwargs):
        lag = iw - jw
        ok = lag >= min_lag
        if max_lag is not None:
            ok = ok & (lag <= max_lag)
        return ok

    try:
        alignment = dtw(
            recharge_norm, precip_norm,
            step_pattern="symmetric2",
            window_type=causal_window,
            window_args={"min_lag": min_lag, "max_lag": max_lag},
            keep_internals=True,
            open_begin=False,
            open_end=False,
        )
    except Exception as e:
        return None, f"dtw failed: {e}"

    event_table = precip_recharge_event_table(
        df, "prcp (mm/day)", "RpSy (m)", alignment, precip_threshold=precip_threshold
    )

    out_path = f"{out_dir}/{site_id}_dtw_event_table.csv"
    event_table.to_csv(out_path, index=False)

    return event_table, "ok"


all_event_tables = {}
failed_wells = {}

for i, site_id in enumerate(eligible_wells["usgs_id"]):
    site_id = str(site_id)
    print(f"\rProcessing well {i+1}/{len(eligible_wells)}...", end="", flush=True)

    out_path = f"dtw_event_tables/{site_id}_dtw_event_table.csv"
    if os.path.exists(out_path):
        all_event_tables[site_id] = pd.read_csv(out_path)
        continue

    table, status = run_dtw_pipeline(site_id)
    if table is not None:
        all_event_tables[site_id] = table
    else:
        failed_wells[site_id] = status

print(f"\n\nCompleted: {len(all_event_tables)} of {len(eligible_wells)} wells")
print(f"Failed/skipped: {len(failed_wells)}")
if failed_wells:
    from collections import Counter
    reasons = Counter(failed_wells.values())
    print("Failure reasons:", dict(reasons))

Total wells with >=5 year records: 163
Processing well 163/163...

Completed: 163 of 163 wells
Failed/skipped: 0


#### Time series visualization

In [7]:
import matplotlib
matplotlib.use('QtAgg')
import matplotlib.pyplot as plt

def plot_dtw_events(site_id, event_table, recharge_dir=recharge_dir, daymet_dir=daymet_dir):
    df_recharge = pd.read_csv(f"{recharge_dir}/{site_id}.csv")
    df_precip = pd.read_csv(f"{daymet_dir}/{site_id}.csv")
    df_precip["Date"] = pd.to_datetime(df_precip['year'] * 1000 + df_precip['yday'], format='%Y%j')
    df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])
    df_recharge.set_index("Date", inplace=True)
    df_precip.set_index("Date", inplace=True)
    df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
    df.drop(columns=["year", "yday"], inplace=True, errors="ignore")

    date_cols = ["recharge_start_date", "recharge_end_date", "precip_start_date", "precip_end_date"]
    event_table[date_cols] = event_table[date_cols].apply(pd.to_datetime)

    fig, ax1 = plt.subplots(figsize=(10, 5))
    color = 'tab:red'
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Recharge (m)', color=color)
    recharge_max = df["RpSy (m)"].max()
    ax1.set_ylim(0, recharge_max * 1.5)
    ax1.plot(df.index, df["RpSy (m)"], color=color)
    ax1.tick_params(axis='y', labelcolor=color)
   
    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Precipitation (mm)', color=color)
    ax2.plot(df.index, df["prcp (mm/day)"], color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    precip_max = df["prcp (mm/day)"].max()
    ax2.set_ylim(precip_max * 1.5, 0)
    axmax = ax1.get_ylim()[1]
    for _, row in event_table.iterrows():
        if pd.notna(row["recharge_start_date"]) and pd.notna(row["recharge_end_date"]):
            x = [row["recharge_end_date"], row["recharge_start_date"],
                 row["precip_start_date"], row["precip_end_date"]]
            y = [0, 0, axmax, axmax]
            ax1.fill(x, y, color='gray', alpha=0.2)
    ax1.set_title(f"Well {site_id}")
    plt.show()


In [8]:
preview_ids = list(all_event_tables.keys())[:2]

for site_id in preview_ids:
    plot_dtw_events(site_id, all_event_tables[site_id])

#### Adding the covariates as defines in Groundwater_x_Precipiation_FIXED.ipynb.

In [9]:
def add_covariates(event_table):
    """
    Adds DUR, AVG, RPR to a DTW-based event table.
    MAG uses precip_total directly (from the DTW event boundaries).
    RECH uses recharge_total directly (raw RpSy, per current guidance).
    """
    df = event_table.copy()

    date_cols = ["precip_start_date", "precip_end_date", "recharge_start_date", "recharge_end_date"]
    df[date_cols] = df[date_cols].apply(pd.to_datetime)

    # DUR: days spanned by the precip event (start to end, inclusive)
    df["DUR"] = (df["precip_end_date"] - df["precip_start_date"]).dt.days + 1
  
    # MAG: using the full precip total from this event's DTW-derived window
    df["MAG"] = df["precip_total"]
    
    # AVG: average daily rate across the event
    df["AVG"] = df["MAG"] / df["DUR"]
   
    # RECH: raw recharge total (Sy not yet applied, per current guidance)
    df["RECH"] = df["recharge_total"]
   
    # RPR: recharge (converted m -> mm) / precip magnitude (mm)
    df["RPR"] = (df["RECH"] * 1000) / df["MAG"]
    return df

#### Shared event filter.Applied everywhere events are used: drops storms below MIN_MAG, and drops events whose implied recharge/precip ratio exceeds MAX_RATIO (the snowmelt/DTW-mismatch artifact identified earlier).


In [10]:
MIN_MAG = 10     # minimum storm magnitude (mm) to include an event
MAX_RATIO = 10   # events with recharge/precip ratio above this are dropped as implausible

def filter_events(event_table, min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    Standard event filter used throughout this notebook:
      - Drops events with precipitation below min_mag
      - Drops events whose implied recharge/precip ratio exceeds
        max_ratio (physically implausible, almost always a sign of
        DTW-mismatched, likely snowmelt-driven, recharge)
    Returns the filtered DataFrame with an added "implied_ratio" column.
    """
    df = event_table.dropna(subset=["MAG", "RECH"]).copy()
    df = df[df["MAG"] >= min_mag]
    df["implied_ratio"] = (df["RECH"] * 1000) / df["MAG"]
    df = df[df["implied_ratio"] <= max_ratio]
    return df


#### Grouping events by season and plotting the covariates. 

In [11]:
def add_season(event_table, date_col="precip_start_date"):
    """
    Adds a season column based on meteorological seasons:
    DJF = Winter, MAM = Spring, JJA = Summer, SON = Fall
    """
    df = event_table.copy()
    month = df[date_col].dt.month
    season_map = {
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Spring", 4: "Spring", 5: "Spring",
        6: "Summer", 7: "Summer", 8: "Summer",
        9: "Fall", 10: "Fall", 11: "Fall",
    }
    df["season"] = month.map(season_map)
    return df


def plot_rpr_by_season(event_table, well_id, x_col="DUR", x_label="Duration (days)",
                         max_dur=5, y_pad=1.15, min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    Four separate panels, one per season, showing RPR vs a chosen
    storm characteristic, points colored by recharge (RECH) magnitude.
    Events with DUR > max_dur are excluded. Y-axis is padded so
    points aren't crowded against the top edge.
    """
    df = filter_events(event_table, min_mag=min_mag, max_ratio=max_ratio)
    df = df.dropna(subset=[x_col, "RPR", "RECH", "DUR"])
    df = df[df["DUR"] <= max_dur]

    seasons = ["Winter", "Spring", "Summer", "Fall"]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    vmin, vmax = df["RECH"].min(), df["RECH"].max()
    y_max = df["RPR"].max() * y_pad

    for ax, season in zip(axes.flat, seasons):
        season_data = df[df["season"] == season]

        scatter = ax.scatter(season_data[x_col], season_data["RPR"],
                              c=season_data["RECH"], cmap="viridis",
                              vmin=vmin, vmax=vmax,
                              alpha=0.6, s=30, edgecolor="none")

        ax.set_xlabel(x_label)
        ax.set_ylabel("RPR")
        ax.set_ylim(0, y_max)
        ax.set_title(f"{season} (n={len(season_data)})")
        ax.grid(alpha=0.3)

    fig.colorbar(scatter, ax=axes, shrink=0.7, label="Recharge (m)")
    fig.suptitle(f"Well {well_id} — RPR vs {x_label}, by Season (colored by recharge, DUR ≤ {max_dur} days)", y=1.02)
    plt.show()

In [12]:
site_id = list(all_event_tables.keys())[0]  
enriched_event_table = add_covariates(all_event_tables[site_id])
enriched_event_table = add_season(enriched_event_table)
plot_rpr_by_season(enriched_event_table, site_id, x_col="DUR", x_label="Duration (days)")
plot_rpr_by_season(enriched_event_table, site_id, x_col="MAG", x_label="Magnitude (mm)")

In [13]:
def plot_recharge_by_season(event_table, well_id, x_col="DUR", x_label="Duration (days)",
                              max_dur=5, y_pad=1.15, min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    Four separate panels, one per season, showing RECHARGE (not RPR)
    vs a chosen storm characteristic. Points colored by MAG.
    Events with DUR > max_dur are excluded.
    """
    df = filter_events(event_table, min_mag=min_mag, max_ratio=max_ratio)
    df = df.dropna(subset=[x_col, "RECH", "MAG", "DUR"])
    df = df[df["DUR"] <= max_dur]

    df["RECH_mm"] = df["RECH"] * 1000  # convert to mm for readability

    seasons = ["Winter", "Spring", "Summer", "Fall"]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    vmin, vmax = df["MAG"].min(), df["MAG"].max()
    y_max = df["RECH_mm"].max() * y_pad

    for ax, season in zip(axes.flat, seasons):
        season_data = df[df["season"] == season]

        scatter = ax.scatter(season_data[x_col], season_data["RECH_mm"],
                              c=season_data["MAG"], cmap="viridis",
                              vmin=vmin, vmax=vmax,
                              alpha=0.6, s=30, edgecolor="none")

        ax.set_xlabel(x_label)
        ax.set_ylabel("Recharge (mm)")
        ax.set_ylim(0, y_max)
        ax.set_title(f"{season} (n={len(season_data)})")
        ax.grid(alpha=0.3)

    fig.colorbar(scatter, ax=axes, shrink=0.7, label="Magnitude (mm)")
    fig.suptitle(f"Well {well_id} — Recharge vs {x_label}, by Season (colored by MAG, DUR ≤ {max_dur} days)", y=1.02)
    plt.show()

In [14]:
site_id = list(all_event_tables.keys())[0]
enriched_event_table = add_covariates(all_event_tables[site_id])
enriched_event_table = add_season(enriched_event_table)

plot_recharge_by_season(enriched_event_table, site_id, x_col="DUR", x_label="Duration (days)")
plot_recharge_by_season(enriched_event_table, site_id, x_col="MAG", x_label="Magnitude (mm)")

In [15]:
def plot_precip_recharge_by_season(event_table, well_id, color_col="DUR", color_label="Duration (days)",
                                      max_dur=5, min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    Four panels, one per season: precipitation (x) vs recharge (y),
    colored by the chosen variable (DUR or MAG).
    Events with DUR > max_dur are excluded.
    """
    df = filter_events(event_table, min_mag=min_mag, max_ratio=max_ratio)
    df = df.dropna(subset=["MAG", "RECH", "DUR", color_col])
    df = df[df["DUR"] <= max_dur]
    df["RECH_mm"] = df["RECH"] * 1000

    seasons = ["Winter", "Spring", "Summer", "Fall"]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    vmin, vmax = df[color_col].min(), df[color_col].max()

    for ax, season in zip(axes.flat, seasons):
        season_data = df[df["season"] == season]

        scatter = ax.scatter(season_data["MAG"], season_data["RECH_mm"],
                              c=season_data[color_col], cmap="viridis",
                              vmin=vmin, vmax=vmax,
                              alpha=0.6, s=30, edgecolor="none")

        ax.set_xlabel("Precipitation (mm)")
        ax.set_ylabel("Recharge (mm)")
        ax.set_title(f"{season} (n={len(season_data)})")
        ax.grid(alpha=0.3)

    fig.colorbar(scatter, ax=axes, shrink=0.7, label=color_label)
    fig.suptitle(f"Well {well_id} — Precip vs Recharge, by Season (colored by {color_label}, DUR ≤ {max_dur} days)", y=1.02)
    plt.show()

#### *** Run this version I think it is best

In [16]:
plot_precip_recharge_by_season(enriched_event_table, site_id, color_col="DUR", color_label="Duration (days)")
plot_precip_recharge_by_season(enriched_event_table, site_id, color_col="MAG", color_label="Magnitude (mm)")

In [17]:

def get_usgs_groundwater(site_id, start_date="2014-01-01", end_date="2024-01-01"):
    """
    Pulls daily mean depth-to-water-level data for one USGS site.
    parameterCd 72019 = depth to water level, feet below land surface
    """
    url = "https://waterservices.usgs.gov/nwis/dv/"
    params = {
        "format": "rdb",
        "sites": site_id,
        "startDT": start_date,
        "endDT": end_date,
        "parameterCd": "72019",
        "siteType": "GW",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()

    lines = resp.text.splitlines()
    data_lines = [l for l in lines if not l.startswith("#")]
    if len(data_lines) < 3:
        print(f"No data returned for site {site_id}")
        return None

    df = pd.read_csv(io.StringIO("\n".join(data_lines)), sep="\t")
    df = df.drop(index=0)  # units-description row right after the header

    mean_col = [c for c in df.columns if c.endswith("00003")]
    if not mean_col:
        print(f"No mean column found for site {site_id}. Columns were: {list(df.columns)}")
        return None
    mean_col = mean_col[0]

    out = df[["site_no", "datetime", mean_col]].copy()
    out = out.rename(columns={mean_col: "depth_to_water_ft"})
    out["datetime"] = pd.to_datetime(out["datetime"])
    out["depth_to_water_ft"] = pd.to_numeric(out["depth_to_water_ft"], errors="coerce")

    return out

#### Cumulative plot of the precipiation/recharge relationship. A linear line indicates that all events are equally responsible for recharge, where as a slightly convex line suggests larger events are disproportionately responsible for recharge. 

In [18]:
def plot_cumulative_precip_recharge(event_table, well_id, season=None,
                                      precip_col="MAG", recharge_col="RECH",
                                      min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    Sorts events from largest to smallest precipitation, then plots
    the cumulative fraction of total precipitation (x) against the
    cumulative fraction of total recharge (y).
    Events below min_mag are excluded, since small storms are prone
    to implausible DTW-driven recharge matches.
    If season is given, filters the event_table to just that season first.
    """
    df = filter_events(event_table, min_mag=min_mag, max_ratio=max_ratio)
    df = df.dropna(subset=[precip_col, recharge_col])

    if season is not None:
        df = df[df["season"] == season]
    if len(df) == 0:
        print(f"No events found for {well_id} in season={season}. Skipping.")
        return

    df = df.sort_values(precip_col, ascending=False).reset_index(drop=True)
    total_precip = df[precip_col].sum()
    total_recharge = df[recharge_col].sum()
    df["cum_precip"] = df[precip_col].cumsum()
    df["cum_recharge"] = df[recharge_col].cumsum()
    df["cum_precip_frac"] = df["cum_precip"] / total_precip
    df["cum_recharge_frac"] = df["cum_recharge"] / total_recharge
    x = np.concatenate(([0], df["cum_precip_frac"].values))
    y = np.concatenate(([0], df["cum_recharge_frac"].values))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(x, y, marker="o", color="black", markersize=4, label="Cumulative curve")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.6, label="1:1 line")
    ax.set_xlabel("Cumulative fraction of total precipitation")
    ax.set_ylabel("Cumulative fraction of total recharge")
    title = f"Well {well_id}"
    if season is not None:
        title += f" — {season} (n={len(df)})"
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


def plot_cumulative_by_season(event_table, well_id, precip_col="MAG", recharge_col="RECH",
                                min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    for season in ["Winter", "Spring", "Summer", "Fall"]:
        plot_cumulative_precip_recharge(event_table, well_id, season=season,
                                          precip_col=precip_col, recharge_col=recharge_col,
                                          min_mag=min_mag, max_ratio=max_ratio)

#### Plot filtering out evens smaller than 10mm 

In [19]:
def plot_before_after_comparison(well_id):
    table = add_covariates(all_event_tables[well_id])

    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))

    for ax, min_mag, title in zip(axes, [0, 10], ["Unfiltered (all events)", "Filtered (MAG ≥ 10mm)"]):
        df = table.dropna(subset=["MAG", "RECH"]).copy()
        df = df[df["MAG"] >= min_mag]
        df = df.sort_values("MAG", ascending=False).reset_index(drop=True)

        if len(df) == 0:
            ax.set_title(f"{title} (no events)")
            continue

        total_precip = df["MAG"].sum()
        total_recharge = df["RECH"].sum()
        df["cum_precip_frac"] = df["MAG"].cumsum() / total_precip
        df["cum_recharge_frac"] = df["RECH"].cumsum() / total_recharge

        x = np.concatenate(([0], df["cum_precip_frac"].values))
        y = np.concatenate(([0], df["cum_recharge_frac"].values))

        ax.plot(x, y, marker="o", color="black", markersize=3)
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.6)
        ax.set_xlabel("Cumulative fraction of precipitation")
        ax.set_ylabel("Cumulative fraction of recharge")
        ax.set_title(f"{title} (n={len(df)})")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)

    fig.suptitle(f"Well {well_id} — Before vs After Filtering", y=1.02)
    fig.tight_layout()
    plt.show()


# Run across several different wells
well_ids_to_check = list(all_event_tables.keys())[:3]

for well_id in well_ids_to_check:
    plot_before_after_comparison(well_id)

In [20]:
def lorenz_curve(values):
    """
    Computes Lorenz curve coordinates and Gini coefficient for one
    variable: x = cumulative fraction of events (smallest to largest),
    y = cumulative fraction of the total those events account for.
    """
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    cum_vals = np.cumsum(sorted_vals)
    cum_frac = cum_vals / cum_vals[-1]
    x = np.concatenate(([0], np.arange(1, n + 1) / n))
    y = np.concatenate(([0], cum_frac))
    trapezoid_func = getattr(np, "trapezoid", None) or np.trapz
    gini = 1 - 2 * trapezoid_func(y, x)
    return x, y, gini


def plot_lorenz(event_table, well_id, value_col, label, season=None,
                  min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    Plots the Lorenz curve for one variable (precip magnitude or
    recharge), events sorted smallest to largest. Excludes events
    below min_mag, since small events are prone to implausible
    DTW-driven recharge matches.
    """
    df = filter_events(event_table, min_mag=min_mag, max_ratio=max_ratio)
    df = df.dropna(subset=[value_col])

    if season is not None:
        df = df[df["season"] == season]
    if len(df) < 2:
        print(f"Too few events for {well_id}, season={season}. Skipping.")
        return None

    x, y, gini = lorenz_curve(df[value_col].values)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(x, y, color="black", linewidth=2)
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.6, label="Perfect equality")
    ax.fill_between(x, y, x, alpha=0.15, color="tab:red")
    ax.set_xlabel(f"Cumulative fraction of {label} events (smallest to largest)")
    ax.set_ylabel(f"Cumulative fraction of total {label}")
    title = f"Well {well_id} — {label} Lorenz Curve\nGini = {gini:.3f}"
    if season is not None:
        title += f" ({season})"
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
    return gini


def plot_lorenz_by_season(event_table, well_id, value_col, label,
                             min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    ginis = {}
    for season in ["Winter", "Spring", "Summer", "Fall"]:
        gini = plot_lorenz(event_table, well_id, value_col, label, season=season,
                             min_mag=min_mag, max_ratio=max_ratio)
        if gini is not None:
            ginis[season] = gini
    return ginis


In [21]:
def compare_filter_impact(well_id, min_mag=10):
    """
    Numerically compares unfiltered vs filtered event tables for one
    well: event counts, Gini coefficients, and the extreme-ratio
    check from before (small events' implied recharge/precip ratio).
    """
    table = add_covariates(all_event_tables[well_id])
    df = table.dropna(subset=["MAG", "RECH"]).copy()

    unfiltered = df.copy()
    filtered = df[df["MAG"] >= min_mag].copy()

    def gini_of(values):
        if len(values) < 2:
            return None
        _, _, g = lorenz_curve(values)
        return g

    unfiltered_precip_gini = gini_of(unfiltered["MAG"].values)
    unfiltered_recharge_gini = gini_of(unfiltered["RECH"].values)
    filtered_precip_gini = gini_of(filtered["MAG"].values)
    filtered_recharge_gini = gini_of(filtered["RECH"].values)

    unfiltered["implied_ratio"] = (unfiltered["RECH"] * 1000) / unfiltered["MAG"]
    filtered["implied_ratio"] = (filtered["RECH"] * 1000) / filtered["MAG"]

    return {
        "well_id": well_id,
        "n_unfiltered": len(unfiltered),
        "n_filtered": len(filtered),
        "unfiltered_recharge_gini": unfiltered_recharge_gini,
        "filtered_recharge_gini": filtered_recharge_gini,
        "unfiltered_max_ratio": unfiltered["implied_ratio"].max(),
        "filtered_max_ratio": filtered["implied_ratio"].max(),
        "unfiltered_median_ratio": unfiltered["implied_ratio"].median(),
        "filtered_median_ratio": filtered["implied_ratio"].median(),
    }


comparison_results = []
for well_id in all_event_tables.keys():
    result = compare_filter_impact(well_id)
    comparison_results.append(result)

comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.describe())

       n_unfiltered   n_filtered  unfiltered_recharge_gini  \
count    163.000000   163.000000                163.000000   
mean    1016.417178   630.846626                  0.623295   
std      384.741790   244.821693                  0.083973   
min      326.000000   205.000000                  0.362944   
25%      735.000000   460.500000                  0.566393   
50%     1050.000000   631.000000                  0.634161   
75%     1290.500000   781.500000                  0.685157   
max     2009.000000  1322.000000                  0.800177   

       filtered_recharge_gini  unfiltered_max_ratio  filtered_max_ratio  \
count              163.000000            163.000000          163.000000   
mean                 0.551610            154.068506           56.660604   
std                  0.090402            222.775124           80.075792   
min                  0.333085             13.419981            4.354730   
25%                  0.492175             40.789164           17.0

#### By worst I mean they were the wells with the most implausible recharge to precipitation ratios. Weirdly they are all in PA. 

In [22]:
worst_wells = ["394430077225001", "404140077354001", "414640077493801", "404556077525101",
               "413026076352901", "402512074414301", "411833075133601", "404708076070701",
               "412020079133901", "443647070552303"]

for well_id in worst_wells:
    if well_id in df_ne_wells["usgs_id"].astype(str).values:
        row = df_ne_wells[df_ne_wells["usgs_id"].astype(str) == well_id].iloc[0]
        print(f"{well_id}: state={row.get('state', 'N/A')}, "
              f"record_length={row.get('record_length', 'N/A'):.1f} yrs")

394430077225001: state=Pennsylvania, record_length=19.0 yrs
404140077354001: state=Pennsylvania, record_length=17.0 yrs
414640077493801: state=Pennsylvania, record_length=25.0 yrs
404556077525101: state=Pennsylvania, record_length=21.0 yrs
413026076352901: state=Pennsylvania, record_length=29.0 yrs
402512074414301: state=New Jersey, record_length=7.0 yrs
411833075133601: state=Pennsylvania, record_length=21.0 yrs
404708076070701: state=Pennsylvania, record_length=26.0 yrs
412020079133901: state=Pennsylvania, record_length=10.0 yrs
443647070552303: state=Maine, record_length=9.0 yrs


In [23]:
worst_id = "394430077225001"
table = add_covariates(all_event_tables[worst_id])
filtered = table[table["MAG"] >= 10].dropna(subset=["MAG", "RECH"]).copy()
filtered["implied_ratio"] = (filtered["RECH"] * 1000) / filtered["MAG"]

top_offenders = filtered.sort_values("implied_ratio", ascending=False).head(10)
print(top_offenders[["precip_start_date", "MAG", "recharge_start_date", "recharge_end_date",
                       "RECH", "n_recharge_days", "implied_ratio"]])

     precip_start_date    MAG recharge_start_date recharge_end_date      RECH  \
607         2015-03-10  10.48          2015-03-10        2015-03-15  6.220558   
544         2014-02-21  10.09          2014-02-21        2014-02-25  4.662625   
956         2021-02-22  10.94          2021-02-22        2021-02-27  4.861097   
1013        2022-04-18  25.49          2022-04-18        2022-04-23  6.138741   
954         2021-02-15  19.08          2021-02-15        2021-02-19  3.855404   
372         2011-02-24  13.27          2011-02-25        2011-03-01  2.642837   
1007        2022-03-07  15.82          2022-03-09        2022-03-14  3.096945   
947         2021-01-01  19.59          2021-01-01        2021-01-05  3.555624   
1004        2022-02-17  14.37          2022-02-17        2022-02-21  2.598281   
214         2008-02-13  15.30          2008-02-13        2008-02-18  2.650323   

      n_recharge_days  implied_ratio  
607                 6     593.564680  
544                 5     462.

#### ****

In [24]:
site_id = list(all_event_tables.keys())[80]
plot_cumulative_precip_recharge(enriched_event_table, site_id)
plot_cumulative_by_season(enriched_event_table, site_id)

In [25]:
def check_small_event_tail(event_table, well_id, n_smallest=10, precip_col="MAG", recharge_col="RECH"):
    """
    Looks at the n_smallest precipitation events for one well, and
    reports their recharge values and the implied RPR-like ratio,
    to check whether small events show disproportionately large recharge.
    """
    df = event_table.dropna(subset=[precip_col, recharge_col]).copy()
    df = df.sort_values(precip_col, ascending=True).head(n_smallest)
    df["implied_ratio"] = (df[recharge_col] * 1000) / df[precip_col]
    return df[["precip_start_date", precip_col, recharge_col, "implied_ratio"]]


results = []
for well_id, table in all_event_tables.items():
    enriched = add_covariates(table)
    small = check_small_event_tail(enriched, well_id)
    if len(small) > 0:
        median_ratio = small["implied_ratio"].median()
        max_ratio = small["implied_ratio"].max()
        results.append({"well_id": well_id, "median_ratio_smallest": median_ratio, "max_ratio_smallest": max_ratio})

tail_check = pd.DataFrame(results)
print(tail_check.describe())
print("\nWells with the most extreme small-event ratios:")
print(tail_check.sort_values("max_ratio_smallest", ascending=False).head(15))

       median_ratio_smallest  max_ratio_smallest
count             163.000000          163.000000
mean                7.857469           58.663032
std                12.110415           73.291053
min                 0.000000            3.603596
25%                 1.882790           16.668597
50%                 4.320608           29.655686
75%                 9.078151           67.757935
max                82.271972          477.019810

Wells with the most extreme small-event ratios:
             well_id  median_ratio_smallest  max_ratio_smallest
113  421213076313301              65.516813          477.019810
85   413346075421301              45.406282          297.975098
41   402512074414301              82.271972          280.290711
99   414640077493801              14.880790          279.650728
51   404140077354001              15.034735          254.230803
63   410155074060201              75.076615          247.886208
54   404556077525101              54.576425          245.84116

In [26]:
site_id = list(all_event_tables.keys())[0]
table = add_covariates(all_event_tables[site_id])
df = table.dropna(subset=["MAG", "RECH"])
df = df[df["MAG"] >= 10]
df["implied_ratio"] = (df["RECH"] * 1000) / df["MAG"]

print(df["implied_ratio"].describe())
print("\nPercentiles:")
for p in [50, 75, 90, 95, 99]:
    print(f"  {p}th: {df['implied_ratio'].quantile(p/100):.2f}")

count    467.000000
mean       3.095250
std        2.844527
min        0.000000
25%        0.826817
50%        2.443517
75%        4.509449
max       20.175790
Name: implied_ratio, dtype: float64

Percentiles:
  50th: 2.44
  75th: 4.51
  90th: 7.11
  95th: 8.37
  99th: 11.51


In [27]:
def filter_implausible_ratios(event_table, max_ratio=MAX_RATIO, min_mag=MIN_MAG):
    """
    Drops events where recharge is more than max_ratio times the
    storm's precipitation -- physically implausible, almost always
    a sign of DTW-mismatched (likely snowmelt-driven) recharge.
    """
    df = event_table.dropna(subset=["MAG", "RECH"]).copy()
    df = df[df["MAG"] >= min_mag]

    df["implied_ratio"] = (df["RECH"] * 1000) / df["MAG"]

    before = len(df)
    filtered = df[df["implied_ratio"] <= max_ratio].copy()
    after = len(filtered)

    print(f"Kept {after} of {before} events (dropped {before - after} with ratio > {max_ratio})")

    return filtered

In [28]:
filtered_event_table = filter_implausible_ratios(enriched_event_table, max_ratio=10, min_mag=10)

Kept 457 of 467 events (dropped 10 with ratio > 10)


In [29]:
worst_well_id = "421213076313301"  # the well with max_ratio_smallest = 477

enriched_worst = add_covariates(all_event_tables[worst_well_id])
small_worst = enriched_worst.dropna(subset=["MAG", "RECH"]).sort_values("MAG").head(15).copy()
small_worst["implied_ratio"] = (small_worst["RECH"] * 1000) / small_worst["MAG"]

print(small_worst[["precip_start_date", "precip_end_date", "MAG",
                    "recharge_start_date", "recharge_end_date", "RECH", "n_recharge_days", "implied_ratio"]])

    precip_start_date precip_end_date   MAG recharge_start_date  \
525        2017-08-10      2017-08-10  2.51          2017-08-10   
254        2013-02-03      2013-02-03  2.51          2013-02-04   
670        2019-08-21      2019-08-21  2.52          2019-08-23   
376        2015-03-13      2015-03-13  2.53          2015-03-13   
168        2011-07-31      2011-07-31  2.53          2011-08-03   
638        2019-04-29      2019-04-29  2.56          2019-04-29   
553        2018-02-09      2018-02-09  2.59          2018-02-09   
721        2020-06-30      2020-06-30  2.60          2020-07-05   
736        2020-10-01      2020-10-01  2.60          2020-10-01   
821        2022-03-02      2022-03-02  2.60          2022-03-02   
835        2022-05-13      2022-05-13  2.61          2022-05-13   
482        2016-12-13      2016-12-13  2.62          2016-12-16   
251        2013-01-11      2013-01-11  2.63          2013-01-12   
559        2018-03-06      2018-03-06  2.65          2018-03-0

In [30]:
def rebuild_master_with_ratio_filter(all_event_tables, max_ratio=10, min_mag=10):
    rows = []
    for well_id, table in all_event_tables.items():
        enriched = add_covariates(table)
        filtered = filter_implausible_ratios(enriched, max_ratio=max_ratio, min_mag=min_mag)

        if len(filtered) == 0:
            continue

        total_precip = filtered["MAG"].sum()
        total_recharge = filtered["RECH"].sum() * 1000

        rows.append({
            "well_id": well_id,
            "n_events": len(filtered),
            "total_precip_mm": total_precip,
            "total_recharge_mm": total_recharge,
            "recharge_efficiency": total_recharge / total_precip if total_precip > 0 else None,
        })

    return pd.DataFrame(rows)


cleaned_summary = rebuild_master_with_ratio_filter(all_event_tables, max_ratio=10)
print(f"Wells with valid data: {len(cleaned_summary)}")

Kept 457 of 467 events (dropped 10 with ratio > 10)
Kept 728 of 729 events (dropped 1 with ratio > 10)
Kept 361 of 376 events (dropped 15 with ratio > 10)
Kept 419 of 423 events (dropped 4 with ratio > 10)
Kept 1212 of 1215 events (dropped 3 with ratio > 10)
Kept 531 of 533 events (dropped 2 with ratio > 10)
Kept 662 of 694 events (dropped 32 with ratio > 10)
Kept 442 of 569 events (dropped 127 with ratio > 10)
Kept 750 of 753 events (dropped 3 with ratio > 10)
Kept 624 of 635 events (dropped 11 with ratio > 10)
Kept 693 of 722 events (dropped 29 with ratio > 10)
Kept 575 of 588 events (dropped 13 with ratio > 10)
Kept 992 of 993 events (dropped 1 with ratio > 10)
Kept 611 of 614 events (dropped 3 with ratio > 10)
Kept 165 of 631 events (dropped 466 with ratio > 10)
Kept 496 of 496 events (dropped 0 with ratio > 10)
Kept 517 of 535 events (dropped 18 with ratio > 10)
Kept 391 of 467 events (dropped 76 with ratio > 10)
Kept 264 of 269 events (dropped 5 with ratio > 10)
Kept 548 of 548 e

#### Ploting the annual trends in precipitation and recharge. A seasonal version is included as well. 

In [31]:
def plot_annual_trends(event_table, well_id, season=None,
                         precip_col="MAG", recharge_col="RECH", date_col="precip_start_date",
                         min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    Two stacked panels: annual total precipitation (top) and annual
    total recharge (bottom), each with a fitted linear trend line.
    """
    df = filter_events(event_table, min_mag=min_mag, max_ratio=max_ratio)
    df = df.dropna(subset=[precip_col, recharge_col])

    if season is not None:
        df = df[df["season"] == season]

    if len(df) == 0:
        print(f"No events found for {well_id}, season={season}. Skipping.")
        return

    df["year"] = df[date_col].dt.year
    yearly = df.groupby("year").agg(
        precip_total=(precip_col, "sum"),
        recharge_total=(recharge_col, "sum"),
    ).reset_index()
    yearly["recharge_mm"] = yearly["recharge_total"] * 1000  # convert to mm

    if len(yearly) < 4:
        print(f"Only {len(yearly)} years of data -- too short for a meaningful trend. Skipping.")
        return

    precip_slope, precip_intercept = np.polyfit(yearly["year"], yearly["precip_total"], 1)
    recharge_slope, recharge_intercept = np.polyfit(yearly["year"], yearly["recharge_mm"], 1)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 9), sharex=True)

    ax1.bar(yearly["year"], yearly["precip_total"], color="tab:blue", alpha=0.6)
    ax1.set_ylabel("Annual precipitation (mm)")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.bar(yearly["year"], yearly["recharge_mm"], color="tab:green", alpha=0.6)
    ax2.set_ylabel("Annual recharge (mm)")
    ax2.set_xlabel("Year")
    ax2.legend()
    ax2.grid(alpha=0.3)

    title = f"Well {well_id} — Annual Precipitation and Recharge Trends"
    if season is not None:
        title += f" ({season})"
    fig.suptitle(title, y=0.98)
    fig.tight_layout()
    plt.show()


def plot_annual_trends_season(event_table, well_id, precip_col="MAG", recharge_col="RECH",
                                   date_col="precip_start_date",
                                   min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    One figure, 4 columns (Winter, Spring, Summer, Fall) x 2 rows
    (precip on top, recharge on bottom), each panel with its own
    linear trend line.
    """
    seasons = ["Winter", "Spring", "Summer", "Fall"]
    fig, axes = plt.subplots(2, 4, figsize=(20, 9), sharey="row")

    filtered_all = filter_events(event_table, min_mag=min_mag, max_ratio=max_ratio)
    filtered_all = filtered_all.dropna(subset=[precip_col, recharge_col])

    for col, season in enumerate(seasons):
        df = filtered_all[filtered_all["season"] == season].copy()

        ax_p = axes[0, col]
        ax_r = axes[1, col]

        if len(df) == 0:
            ax_p.set_title(f"{season} (no data)")
            continue

        df["year"] = df[date_col].dt.year
        yearly = df.groupby("year").agg(
            precip_total=(precip_col, "sum"),
            recharge_total=(recharge_col, "sum"),
        ).reset_index()
        yearly["recharge_mm"] = yearly["recharge_total"] * 1000

        if len(yearly) < 4:
            ax_p.set_title(f"{season} (only {len(yearly)} yrs)")
            ax_p.bar(yearly["year"], yearly["precip_total"], color="tab:blue", alpha=0.6)
            ax_r.bar(yearly["year"], yearly["recharge_mm"], color="tab:green", alpha=0.6)
            continue

        precip_slope, precip_intercept = np.polyfit(yearly["year"], yearly["precip_total"], 1)
        recharge_slope, recharge_intercept = np.polyfit(yearly["year"], yearly["recharge_mm"], 1)

        ax_p.bar(yearly["year"], yearly["precip_total"], color="tab:blue", alpha=0.6)
        ax_p.set_title(f"{season}\nslope: {precip_slope:+.2f} mm/yr")
        ax_p.grid(alpha=0.3)

        ax_r.bar(yearly["year"], yearly["recharge_mm"], color="tab:green", alpha=0.6)
        ax_r.set_title(f"slope: {recharge_slope:+.2f} mm/yr")
        ax_r.set_xlabel("Year")
        ax_r.grid(alpha=0.3)

    axes[0, 0].set_ylabel("Annual precipitation (mm)")
    axes[1, 0].set_ylabel("Annual recharge (mm)")

    fig.suptitle(f"Well {well_id} — Seasonal Precipitation and Recharge Trends", y=1.0)
    fig.tight_layout()
    plt.show()

In [32]:
site_id = list(all_event_tables.keys())[27]
enriched_event_table = add_covariates(all_event_tables[site_id])
enriched_event_table = add_season(enriched_event_table)

plot_annual_trends(enriched_event_table, site_id)
plot_annual_trends_season(enriched_event_table, site_id)

C:\Users\romin\AppData\Local\Temp\ipykernel_33760\1216424692.py:36: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax1.legend()
C:\Users\romin\AppData\Local\Temp\ipykernel_33760\1216424692.py:42: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax2.legend()


In [33]:
import pymannkendall as mk

def compute_well_trend(event_table, value_col, date_col="precip_start_date"):
    """
    Runs Mann-Kendall on annual totals for one well. Returns slope
    and significance, or None if too little data.
    """
    df = event_table.dropna(subset=[value_col]).copy()
    df["year"] = df[date_col].dt.year
    yearly = df.groupby("year")[value_col].sum().reset_index()

    if len(yearly) < 4:
        return None

    result = mk.original_test(yearly[value_col].values)
    return {"slope": result.slope, "p": result.p, "trend": result.trend}


well_trends = []

for well_id in all_event_tables.keys():
    table = add_covariates(all_event_tables[well_id])

    precip_trend = compute_well_trend(table, "MAG")
    recharge_trend = compute_well_trend(table, "RECH")

    if precip_trend is None or recharge_trend is None:
        continue

    lat = df_ne_wells.loc[df_ne_wells["usgs_id"].astype(str) == well_id, "lat"].values
    lon = df_ne_wells.loc[df_ne_wells["usgs_id"].astype(str) == well_id, "lon"].values

    if len(lat) == 0:
        continue

    well_trends.append({
        "well_id": well_id,
        "lat": lat[0],
        "lon": lon[0],
        "precip_slope": precip_trend["slope"],
        "precip_p": precip_trend["p"],
        "recharge_slope": recharge_trend["slope"],
        "recharge_p": recharge_trend["p"],
    })

well_trends_df = pd.DataFrame(well_trends)
print(f"Wells with computable trends: {len(well_trends_df)}")
print(well_trends_df.head())

Wells with computable trends: 163
           well_id        lat        lon  precip_slope  precip_p  \
0  390211074505502  39.036502 -74.848224     -0.406000  1.000000   
1  391145074520401  39.195948 -74.867392      5.303333  0.397587   
2  391621074435401  39.272615 -74.731274     13.780000  0.372691   
3  392232074234403  39.395672 -74.625158     10.470000  0.246387   
4  392731075092401  39.459004 -75.157683      5.312800  0.186426   

   recharge_slope  recharge_p  
0        0.019348    0.766525  
1        0.022939    0.310046  
2        0.189474    0.192616  
3        0.100176    0.099509  
4        0.035847    0.018452  


In [34]:
def plot_trend_map(well_trends_df, value_col, title, states_shp=state_boundaries):
    states = gpd.read_file(states_shp)
    ne = states[states["NAME"].isin(NE_states)].copy()
    ne["geometry"] = ne["geometry"].buffer(0)
    ne = ne.set_crs(epsg=4326, allow_override=True)

    fig, ax = plt.subplots(figsize=(10, 10))
    ne.boundary.plot(ax=ax, color="black", linewidth=0.6)

    vmax = well_trends_df[value_col].abs().max()
    scatter = ax.scatter(
        well_trends_df["lon"], well_trends_df["lat"],
        c=well_trends_df[value_col], cmap="RdBu",
        vmin=-vmax, vmax=vmax,
        s=80, edgecolor="black", linewidth=0.5, zorder=3
    )

    cbar = fig.colorbar(scatter, ax=ax, shrink=0.7)
    cbar.set_label(f"{value_col} (mm/yr)")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title)
    fig.tight_layout()
    plt.show()


# plot_trend_map(well_trends_df, "precip_slope", "Precipitation Trend by Well (mm/yr)")

In [35]:
site_id = list(all_event_tables.keys())[1]

precip_gini = plot_lorenz(enriched_event_table, site_id, "MAG", "Precipitation")
recharge_gini = plot_lorenz(enriched_event_table, site_id, "RECH", "Recharge")

print(f"Precipitation Gini: {precip_gini:.3f}")
print(f"Recharge Gini: {recharge_gini:.3f}")

Precipitation Gini: 0.374
Recharge Gini: 0.697


In [36]:


precip_ginis = plot_lorenz_by_season(enriched_event_table, site_id, "MAG", "Precipitation")
recharge_ginis = plot_lorenz_by_season(enriched_event_table, site_id, "RECH", "Recharge")

print("Precipitation Gini by season:")
for season, gini in precip_ginis.items():
    print(f"  {season}: {gini:.3f}")

print("Recharge Gini by season:")
for season, gini in recharge_ginis.items():
    print(f"  {season}: {gini:.3f}")

Precipitation Gini by season:
  Winter: 0.310
  Spring: 0.334
  Summer: 0.391
  Fall: 0.395
Recharge Gini by season:
  Winter: 0.454
  Spring: 0.589
  Summer: 0.860
  Fall: 0.688


#### Gini Summary overall + seasonal

In [37]:
def compute_gini_summary(event_table, well_id, plot=True, min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    Computes precip and recharge Gini coefficients, both overall and
    for each season
    """
    rows = []
    scopes = [("Overall", None)] + [(s, s) for s in ["Winter", "Spring", "Summer", "Fall"]]
    for label, season in scopes:
        df = filter_events(event_table, min_mag=min_mag, max_ratio=max_ratio)
        if season is not None:
            df = df[df["season"] == season]

        precip_df = df.dropna(subset=["MAG"])
        recharge_df = df.dropna(subset=["RECH"])

        if len(precip_df) >= 2:
            if plot:
                precip_gini = plot_lorenz(event_table, well_id, "MAG", "Precipitation", season=season, min_mag=min_mag, max_ratio=max_ratio)
            else:
                _, _, precip_gini = lorenz_curve(precip_df["MAG"].values)
        else:
            precip_gini = None

        if len(recharge_df) >= 2:
            if plot:
                recharge_gini = plot_lorenz(event_table, well_id, "RECH", "Recharge", season=season, min_mag=min_mag, max_ratio=max_ratio)
            else:
                _, _, recharge_gini = lorenz_curve(recharge_df["RECH"].values)
        else:
            recharge_gini = None

        rows.append({
            "well_id": well_id,
            "scope": label,
            "n_events": len(df),
            "precip_gini": precip_gini,
            "recharge_gini": recharge_gini,
        })
    return pd.DataFrame(rows)

In [38]:
all_gini_summaries = []
for well_id in all_event_tables.keys():
    table = add_covariates(all_event_tables[well_id])
    table = add_season(table)
    summary = compute_gini_summary(table, well_id, plot=False)
    all_gini_summaries.append(summary)

gini_comparison = pd.concat(all_gini_summaries, ignore_index=True)
print(f"Total rows: {len(gini_comparison)}")
gini_comparison

Total rows: 815


,well_id,scope,n_events,precip_gini,recharge_gini
0,390211074505502,Overall,457,0.346444,0.587890
1,390211074505502,Winter,121,0.299947,0.484056
2,390211074505502,Spring,118,0.321719,0.523468
3,390211074505502,Summer,119,0.346401,0.679231
4,390211074505502,Fall,99,0.401401,0.618917
...,...,...,...,...,...
810,453629068531801,Overall,655,0.337159,0.718396
811,453629068531801,Winter,156,0.271985,0.709284
812,453629068531801,Spring,145,0.328724,0.687567
813,453629068531801,Summer,181,0.343435,0.745340


In [39]:
def compute_well_gini_filtered(min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    Computes precip and recharge Gini coefficients for every well 
    and attaches lat/lon.
    """
    rows = []

    for well_id in all_event_tables.keys():
        table = add_covariates(all_event_tables[well_id])
        df = filter_events(table, min_mag=min_mag, max_ratio=max_ratio)

        if len(df) < 2:
            continue

        _, _, precip_gini = lorenz_curve(df["MAG"].values)
        _, _, recharge_gini = lorenz_curve(df["RECH"].values)

        lat = df_ne_wells.loc[df_ne_wells["usgs_id"].astype(str) == well_id, "lat"].values
        lon = df_ne_wells.loc[df_ne_wells["usgs_id"].astype(str) == well_id, "lon"].values

        if len(lat) == 0:
            continue

        rows.append({
            "well_id": well_id,
            "lat": lat[0],
            "lon": lon[0],
            "n_events": len(df),
            "precip_gini": precip_gini,
            "recharge_gini": recharge_gini,
        })

    return pd.DataFrame(rows)


well_gini_df = compute_well_gini_filtered(min_mag=MIN_MAG, max_ratio=MAX_RATIO)
print(f"Wells with computable Gini: {len(well_gini_df)}")
print(well_gini_df.head())

Wells with computable Gini: 163
           well_id        lat        lon  n_events  precip_gini  recharge_gini
0  390211074505502  39.036502 -74.848224       457     0.346444       0.587890
1  391145074520401  39.195948 -74.867392       728     0.349660       0.542135
2  391621074435401  39.272615 -74.731274       361     0.342303       0.439625
3  392232074234403  39.395672 -74.625158       419     0.337874       0.476541
4  392731075092401  39.459004 -75.157683      1212     0.344400       0.433206


#### Map of the NE with wells colored by their gini values. 

In [40]:
def plot_gini_map(well_gini_df, value_col, title, states_shp=state_boundaries):
    states = gpd.read_file(states_shp)
    ne = states[states["NAME"].isin(NE_states)].copy()
    ne["geometry"] = ne["geometry"].buffer(0)
    ne = ne.set_crs(epsg=4326, allow_override=True)

    fig, ax = plt.subplots(figsize=(10, 10))
    ne.boundary.plot(ax=ax, color="black", linewidth=0.6)

    # Scale to the ACTUAL range of your data, not the theoretical 0-1
    vmin = well_gini_df[value_col].min()
    vmax = well_gini_df[value_col].max()

    scatter = ax.scatter(
        well_gini_df["lon"], well_gini_df["lat"],
        c=well_gini_df[value_col], cmap="viridis",
        vmin=vmin, vmax=vmax,
        s=80, edgecolor="black", linewidth=0.5, zorder=3
    )

    cbar = fig.colorbar(scatter, ax=ax, shrink=0.7)
    cbar.set_label(f"{value_col} (Gini coefficient)")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"{title}\n(range: {vmin:.2f} to {vmax:.2f})")
    fig.tight_layout()
    plt.show()


plot_gini_map(well_gini_df, "precip_gini", "Precipitation Gini by Well (MAG ≥ 10mm)")
plot_gini_map(well_gini_df, "recharge_gini", "Recharge Gini by Well (MAG ≥ 10mm)")

#### Dont really know what this data means in this context

In [41]:
def plot_gini_comparison_scatter(well_gini_df, zoom=True):
    """
    One dot per well: x = precipitation Gini, y = recharge Gini.
    If zoom=True, axes are scaled to the actual data range instead
    of the full theoretical 0-1 range, so the real spread is visible.
    """
    df = well_gini_df.dropna(subset=["precip_gini", "recharge_gini"])

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(df["precip_gini"], df["recharge_gini"], alpha=0.5, s=50, color="tab:purple")

    if zoom:
        x_min, x_max = df["precip_gini"].min(), df["precip_gini"].max()
        y_min, y_max = df["recharge_gini"].min(), df["recharge_gini"].max()

        pad_x = (x_max - x_min) * 0.1
        pad_y = (y_max - y_min) * 0.1

        xlim = (x_min - pad_x, x_max + pad_x)
        ylim = (y_min - pad_y, y_max + pad_y)

        # extend the 1:1 line just far enough to be visible in this zoomed range
        line_min = min(xlim[0], ylim[0])
        line_max = max(xlim[1], ylim[1])
        ax.plot([line_min, line_max], [line_min, line_max],
                linestyle="--", color="gray", alpha=0.6, label="1:1 line")

        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
    else:
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.6, label="1:1 line")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

    ax.set_xlabel("Precipitation Gini")
    ax.set_ylabel("Recharge Gini")
    ax.set_title("Recharge Concentration vs Precipitation Concentration, per well")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_gini_comparison_scatter(well_gini_df, zoom=True)

#### Not a fan of this

In [42]:
def plot_precip_vs_recharge(event_table, well_id, color_col="DUR", color_label="Duration (days)",
                              min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    df = filter_events(event_table, min_mag=min_mag, max_ratio=max_ratio)
    df = df.dropna(subset=[color_col])
    df["RECH_mm"] = df["RECH"] * 1000
    df = df[df["RECH_mm"] > 0]  
    fig, ax = plt.subplots(figsize=(9, 7))
    scatter = ax.scatter(df["MAG"], df["RECH_mm"], c=df[color_col], cmap="plasma", alpha=0.6, s=30)
    ax.set_xscale("log")
    ax.set_yscale("log")

    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label(color_label)
    ax.set_xlabel("Precipitation (mm, log scale)")
    ax.set_ylabel("Recharge (mm, log scale)")
    ax.set_title(f"Well {well_id} — Precip vs Recharge (log-log), colored by {color_label}")
    ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
    plt.show()

#### Precip v Recharge colored by mag and dur

In [43]:
site_id = list(all_event_tables.keys())[0]
enriched_event_table = add_covariates(all_event_tables[site_id])
enriched_event_table = add_season(enriched_event_table)

plot_precip_vs_recharge(enriched_event_table, site_id, color_col="DUR", color_label="Duration (days)")
plot_precip_vs_recharge(enriched_event_table, site_id, color_col="MAG", color_label="Magnitude (mm)")

#### Find groundwater level day before a storm. 

In [44]:
def add_water_table_before_storm(event_table, gw_head_df, date_col="precip_start_date",
                                    depth_col="depth_to_water_ft"):
    """
    For each event, finds the USGS depth-to-water reading from the
    day immediately before the storm started. Converts to meters.
    """
    df = event_table.copy()

    depths = []
    for _, row in df.iterrows():
        target_date = row[date_col] - pd.Timedelta(days=1)

        # find the closest available reading on or before the target date,
        # in case the exact day-before is missing
        match = gw_head_df[gw_head_df["date"] <= target_date].sort_values("date")

        if len(match) == 0:
            depths.append(None)
        else:
            depths.append(match.iloc[-1][depth_col])

    df["WT_depth_before_ft"] = depths
    df["WT_depth_before_m"] = df["WT_depth_before_ft"] * 0.3048

    return df

In [45]:
site_id = list(all_event_tables.keys())[0]
enriched_event_table = add_covariates(all_event_tables[site_id])
enriched_event_table = add_season(enriched_event_table)

gw_head = get_usgs_groundwater(
    site_id,
    start_date=enriched_event_table["precip_start_date"].min().strftime("%Y-%m-%d"),
    end_date="2024-01-01"
)

if gw_head is not None and not gw_head.empty:
    gw_head = gw_head.rename(columns={"datetime": "date"})
    enriched_event_table = add_water_table_before_storm(enriched_event_table, gw_head)
    print(enriched_event_table[["precip_start_date", "MAG", "RECH", "WT_depth_before_m"]].dropna(subset=["WT_depth_before_m"]).head(10))
else:
    print(f"No head data available for well {site_id}.")

   precip_start_date     MAG      RECH  WT_depth_before_m
1         2008-10-27    9.35  0.003624           3.124200
2         2008-11-04   38.09  0.076560           3.166872
3         2008-11-08    3.96  0.030898           3.099816
4         2008-11-13   59.30  0.096276           3.084576
5         2008-11-17    3.80  0.043268           2.980944
6         2008-11-24   26.02  0.033531           2.977896
7         2008-11-29   24.92  0.053455           2.941320
8         2008-12-06    2.72  0.044755           2.889504
9         2008-12-10  116.68  0.494895           2.886456
10        2008-12-15   23.21  0.156405           2.286000


#### Precip v Recharge colored by dep day before storm.

In [46]:
plot_precip_vs_recharge(enriched_event_table, site_id, color_col="WT_depth_before_m", color_label="Water table depth before storm (m)")

#### Kumaraswamy Curve fit for individual wells and all

In [47]:
from scipy.optimize import curve_fit

def kumaraswamy_cdf(x, a, b):
    """Kumaraswamy CDF: F(x) = 1 - (1 - x^a)^b"""
    return 1 - (1 - np.clip(x, 0, 1) ** a) ** b


def fit_kumaraswamy(x_data, y_data):
    """
    Fits a and b parameters to match the observed cumulative curve.
    Returns (a, b) or None if the fit fails.
    """
    try:
        popt, _ = curve_fit(kumaraswamy_cdf, x_data, y_data, p0=[1, 1], maxfev=5000)
        return popt
    except Exception:
        return None


def plot_kumaraswamy_fit(event_table, well_id, precip_col="MAG", recharge_col="RECH", min_mag=10):
    """
    Fits a Kumaraswamy curve to this well's precip-vs-recharge
    cumulative curve (same data as the original cumulative plot),
    and overlays the fitted curve for comparison.
    """
    df = event_table.dropna(subset=[precip_col, recharge_col]).copy()
    df = df[df[precip_col] >= min_mag]
    df = df.sort_values(precip_col, ascending=False).reset_index(drop=True)

    total_precip = df[precip_col].sum()
    total_recharge = df[recharge_col].sum()
    df["cum_precip_frac"] = df[precip_col].cumsum() / total_precip
    df["cum_recharge_frac"] = df[recharge_col].cumsum() / total_recharge

    x_data = np.concatenate(([0], df["cum_precip_frac"].values))
    y_data = np.concatenate(([0], df["cum_recharge_frac"].values))

    params = fit_kumaraswamy(x_data, y_data)
    if params is None:
        print(f"Fit failed for well {well_id}")
        return None

    a, b = params
    x_smooth = np.linspace(0, 1, 100)
    y_smooth = kumaraswamy_cdf(x_smooth, a, b)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(x_data, y_data, "o", color="black", markersize=3, alpha=0.5, label="Actual cumulative curve")
    ax.plot(x_smooth, y_smooth, color="red", linewidth=2, label=f"Kumaraswamy fit (a={a:.2f}, b={b:.2f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.5, label="1:1 line")

    ax.set_xlabel("Cumulative fraction of precipitation")
    ax.set_ylabel("Cumulative fraction of recharge")
    ax.set_title(f"Well {well_id} — Kumaraswamy Fit")
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

    return a, b

In [48]:
site_id = list(all_event_tables.keys())[0]
enriched_event_table = add_covariates(all_event_tables[site_id])
a, b = plot_kumaraswamy_fit(enriched_event_table, site_id)

In [49]:
kuma_results = []

for well_id in all_event_tables.keys():
    table = add_covariates(all_event_tables[well_id])
    df = filter_events(table, min_mag=MIN_MAG, max_ratio=MAX_RATIO)

    if len(df) < 5:
        continue

    df = df.sort_values("MAG", ascending=False).reset_index(drop=True)
    total_precip = df["MAG"].sum()
    total_recharge = df["RECH"].sum()
    x_data = np.concatenate(([0], (df["MAG"].cumsum() / total_precip).values))
    y_data = np.concatenate(([0], (df["RECH"].cumsum() / total_recharge).values))

    params = fit_kumaraswamy(x_data, y_data)
    if params is not None:
        kuma_results.append({"well_id": well_id, "a": params[0], "b": params[1]})

kuma_df = pd.DataFrame(kuma_results)
print(f"Successfully fit: {len(kuma_df)} of {len(all_event_tables)} wells")
print(kuma_df.describe())

Successfully fit: 163 of 163 wells
                a           b
count  163.000000  163.000000
mean     0.957019    1.062076
std      0.160726    0.138372
min      0.508176    0.636279
25%      0.859309    0.960723
50%      0.941168    1.054437
75%      1.041889    1.163102
max      1.783093    1.408203


In [50]:
pa_wells = df_ne_wells[df_ne_wells["state"] == "Pennsylvania"]["usgs_id"].astype(str).tolist()
pa_wells_in_data = [w for w in pa_wells if w in all_event_tables]

print(f"PA wells with event tables: {len(pa_wells_in_data)}")

PA wells with event tables: 39


In [51]:
def fit_kumaraswamy_curve(event_table, precip_col="MAG", recharge_col="RECH",
                            min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    Fits Kumaraswamy parameters for one well's cumulative curve,
    without plotting. Returns (a, b) or None.
    """
    df = filter_events(event_table, min_mag=min_mag, max_ratio=max_ratio)

    if len(df) < 5:
        return None

    df = df.sort_values(precip_col, ascending=False).reset_index(drop=True)
    total_precip = df[precip_col].sum()
    total_recharge = df[recharge_col].sum()
    x_data = np.concatenate(([0], (df[precip_col].cumsum() / total_precip).values))
    y_data = np.concatenate(([0], (df[recharge_col].cumsum() / total_recharge).values))

    return fit_kumaraswamy(x_data, y_data)


def plot_all_kumaraswamy_curves(well_ids, title="Kumaraswamy Fits by Well"):
    """
    One figure, every well's fitted curve overlaid as its own colored line.
    """
    fig, ax = plt.subplots(figsize=(9, 8))
    x_smooth = np.linspace(0, 1, 200)

    # Handle both old and new matplotlib colormap access
    try:
        cmap = plt.colormaps.get_cmap("tab20")
    except AttributeError:
        cmap = plt.cm.get_cmap("tab20")

    n = max(len(well_ids), 1)

    for i, well_id in enumerate(well_ids):
        table = add_covariates(all_event_tables[well_id])
        params = fit_kumaraswamy_curve(table)
        if params is None:
            continue

        a, b = params
        y_smooth = kumaraswamy_cdf(x_smooth, a, b)
        color = cmap(i / n)
        ax.plot(x_smooth, y_smooth, color=color, linewidth=1.5,
                label=f"{well_id} (a={a:.2f}, b={b:.2f})")

    ax.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.4, label="1:1 line")

    ax.set_xlabel("x")
    ax.set_ylabel("Cumulative fraction of recharge")
    ax.set_title(title)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=6, loc="upper left", bbox_to_anchor=(1.02, 1))
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

In [52]:
pa_wells = df_ne_wells[df_ne_wells["state"] == "Pennsylvania"]["usgs_id"].astype(str).tolist()
pa_wells_in_data = [w for w in pa_wells if w in all_event_tables]

plot_all_kumaraswamy_curves(pa_wells_in_data, title="Kumaraswamy Fits — Pennsylvania Wells")

In [53]:
nj_wells = df_ne_wells[df_ne_wells["state"] == "New Jersey"]["usgs_id"].astype(str).tolist()
nj_wells_in_data = [w for w in nj_wells if w in all_event_tables]

plot_all_kumaraswamy_curves(nj_wells_in_data, title="Kumaraswamy Fits — New Jersey Wells")

#### Find soil properties and saved them from SoilGrids

In [54]:
def get_soil_properties_cached(lat, lon, properties=("clay", "sand", "silt"), depth="0-5cm",
                                  cache_dir="soil_cache", retries=3, base_delay=2):
    """
    Queries SoilGrids for one location
    """
    os.makedirs(cache_dir, exist_ok=True)
    cache_key = f"{lat:.6f}_{lon:.6f}"
    cache_path = os.path.join(cache_dir, f"{cache_key}.json")

    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            return json.load(f)

    base_url = "https://rest.isric.org/soilgrids/v2.0/properties/query"
    params = {
        "lon": lon, "lat": lat,
        "property": list(properties), "depth": depth, "value": "mean",
    }

    for attempt in range(retries):
        resp = requests.get(base_url, params=params, timeout=30)

        if resp.status_code == 429:
            wait = base_delay * (attempt + 1)
            print(f"Rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue

        if resp.status_code != 200:
            print(f"Failed for ({lat}, {lon}): status {resp.status_code}")
            return None

        data = resp.json()
        result = {}
        for layer in data.get("properties", {}).get("layers", []):
            prop_name = layer["name"]
            for depth_entry in layer.get("depths", []):
                if depth_entry["label"] == depth:
                    result[prop_name] = depth_entry["values"].get("mean")

        with open(cache_path, "w") as f:
            json.dump(result, f)

        return result

    print(f"Gave up on ({lat}, {lon}) after {retries} attempts.")
    return None

In [55]:
soil_gini_rows = []

# If soil_gini_df already exists from a previous run, don't refetch those wells
already_done = set(soil_gini_df["well_id"]) if "soil_gini_df" in dir() else set()

for i, row in well_gini_df.iterrows():
    well_id = row["well_id"]

    if well_id in already_done:
        continue  # already have this one, skip entirely

    well_info = df_ne_wells[df_ne_wells["usgs_id"].astype(str) == well_id]
    if len(well_info) == 0:
        continue

    lat, lon = well_info.iloc[0]["lat"], well_info.iloc[0]["lon"]

    print(f"[{i+1}/{len(well_gini_df)}] {well_id}...")
    soil = get_soil_properties_cached(lat, lon, properties=("clay", "sand", "silt"))

    if soil is None or any(v is None for v in soil.values()):
        continue

    soil_gini_rows.append({
        "well_id": well_id,
        "clay": soil["clay"] / 10,
        "sand": soil["sand"] / 10,
        "silt": soil["silt"] / 10,
        "recharge_gini": row["recharge_gini"],
        "precip_gini": row["precip_gini"],
    })

    time.sleep(1.5)

new_soil_df = pd.DataFrame(soil_gini_rows)

# Combine with whatever was already there, if anything
if "soil_gini_df" in dir() and len(soil_gini_df) > 0:
    soil_gini_df = pd.concat([soil_gini_df, new_soil_df], ignore_index=True)
else:
    soil_gini_df = new_soil_df

print(f"\nWells with soil + Gini data: {len(soil_gini_df)}")

[1/163] 390211074505502...
[2/163] 391145074520401...
[3/163] 391621074435401...
[4/163] 392232074234403...
[5/163] 392731075092401...
[6/163] 392732075092401...
[7/163] 392920074570001...
[8/163] 393232074263903...
[9/163] 393246075012701...
[10/163] 393749074550901...
[11/163] 394106074362501...
[12/163] 394236075272101...
[13/163] 394354075025901...
[14/163] 394422074430903...
[15/163] 394430077225001...
[16/163] 394440074593101...
[17/163] 394452074281901...
[18/163] 394742074142002...
[19/163] 395034074112101...
[20/163] 395122074301702...
[21/163] 395150074284201...
[22/163] 395450075485401...
[23/163] 395512075293701...
[24/163] 395920079021501...
[25/163] 395928074502701...
[26/163] 400120074265401...
[27/163] 400148074352101...
[28/163] 400209077183301...
[29/163] 400217078281901...
[30/163] 400229075104601...
[31/163] 400808075210401...
[32/163] 400916076492301...
[33/163] 401552074501801...
[34/163] 401637076071501...
[35/163] 401753074483501...
[36/163] 401834074515501...
[

#### Soil triangle colored by gini values

In [71]:
def ternary_to_cartesian(clay, sand, silt):
    """
    Converts clay/sand/silt percentages (summing to 100) into 2D
    x,y coordinates for plotting on a triangular (ternary) diagram.
    Standard USDA convention: clay at top, sand at bottom-left,
    silt at bottom-right.
    """
    total = clay + sand + silt
    clay_frac = clay / total
    sand_frac = sand / total
    silt_frac = silt / total

    x = 0.5 * (2 * silt_frac + clay_frac) / (clay_frac + sand_frac + silt_frac)
    y = (np.sqrt(3) / 2) * clay_frac / (clay_frac + sand_frac + silt_frac)

    return x, y


def plot_soil_texture_triangle(soil_gini_df, gini_col="recharge_gini"):
    fig, ax = plt.subplots(figsize=(9, 8))

    # Outer triangle
    triangle_x = [0, 1, 0.5, 0]
    triangle_y = [0, 0, np.sqrt(3)/2, 0]
    ax.plot(triangle_x, triangle_y, color="black", linewidth=1.5)

    # Gridlines at 10% intervals, parallel to each of the three sides
    for pct in range(10, 100, 10):
        # Constant CLAY line: parallel to the sand-silt (bottom) side
        p1 = ternary_to_cartesian(pct, 100 - pct, 0)
        p2 = ternary_to_cartesian(pct, 0, 100 - pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.5, alpha=0.6)

        # Constant SAND line: parallel to the clay-silt (right) side
        p1 = ternary_to_cartesian(100 - pct, pct, 0)
        p2 = ternary_to_cartesian(0, pct, 100 - pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.5, alpha=0.6)

        # Constant SILT line: parallel to the clay-sand (left) side
        p1 = ternary_to_cartesian(100 - pct, 0, pct)
        p2 = ternary_to_cartesian(0, 100 - pct, pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.5, alpha=0.6)

    # Corner labels
    ax.text(-0.03, -0.03, "Sand", fontsize=11, ha="right", va="top")
    ax.text(1.03, -0.03, "Silt", fontsize=11, ha="left", va="top")
    ax.text(0.5, np.sqrt(3)/2 + 0.03, "Clay", fontsize=11, ha="center", va="bottom")

    # Plot wells
    x_coords, y_coords = [], []
    for _, row in soil_gini_df.iterrows():
        x, y = ternary_to_cartesian(row["clay"], row["sand"], row["silt"])
        x_coords.append(x)
        y_coords.append(y)

    scatter = ax.scatter(x_coords, y_coords, c=soil_gini_df[gini_col],
                          cmap="viridis", s=60, edgecolor="black", linewidth=0.5, zorder=3)

    cbar = fig.colorbar(scatter, ax=ax, shrink=0.7)
    cbar.set_label(gini_col)

    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, np.sqrt(3)/2 + 0.1)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"Soil Texture Triangle, colored by {gini_col}")
    fig.tight_layout()
    plt.show()

In [72]:
plot_soil_texture_triangle(soil_gini_df, gini_col="recharge_gini")

In [73]:
test_row = df_ne_wells.iloc[0]
raw_response = requests.get(
    "https://rest.isric.org/soilgrids/v2.0/properties/query",
    params={"lon": test_row["lon"], "lat": test_row["lat"],
            "property": ["clay", "sand", "silt"], "depth": "0-5cm", "value": "mean"},
    timeout=30
).json()

print(raw_response["properties"]["layers"])

[{'name': 'clay', 'unit_measure': {'d_factor': 10, 'mapped_units': 'g/kg', 'target_units': '%', 'uncertainty_unit': ''}, 'depths': [{'range': {'top_depth': 0, 'bottom_depth': 5, 'unit_depth': 'cm'}, 'label': '0-5cm', 'values': {'mean': 80}}]}, {'name': 'sand', 'unit_measure': {'d_factor': 10, 'mapped_units': 'g/kg', 'target_units': '%', 'uncertainty_unit': ''}, 'depths': [{'range': {'top_depth': 0, 'bottom_depth': 5, 'unit_depth': 'cm'}, 'label': '0-5cm', 'values': {'mean': 722}}]}, {'name': 'silt', 'unit_measure': {'d_factor': 10, 'mapped_units': 'g/kg', 'target_units': '%', 'uncertainty_unit': ''}, 'depths': [{'range': {'top_depth': 0, 'bottom_depth': 5, 'unit_depth': 'cm'}, 'label': '0-5cm', 'values': {'mean': 198}}]}]


In [75]:
soil_gini_df["texture_sum"] = soil_gini_df["clay"] + soil_gini_df["sand"] + soil_gini_df["silt"]
print(soil_gini_df["texture_sum"].describe())

count    131.000000
mean     100.003053
std        0.053979
min       99.900000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.100000
Name: texture_sum, dtype: float64


In [76]:
def build_seasonal_soil_gini(gini_comparison, soil_texture_lookup):
    """
    Merges seasonal Gini values with soil texture data for each well.
    soil_texture_lookup should be a DataFrame with well_id, clay, sand, silt
    (the soil data alone, without any Gini column attached).
    """
    merged = gini_comparison.merge(soil_texture_lookup, on="well_id", how="inner")
    return merged


# Extract just the soil texture columns (well_id, clay, sand, silt) from what you already have
soil_texture_lookup = soil_gini_df[["well_id", "clay", "sand", "silt"]].drop_duplicates()

seasonal_soil_gini = build_seasonal_soil_gini(gini_comparison, soil_texture_lookup)
print(seasonal_soil_gini["scope"].value_counts())

scope
Overall    131
Winter     131
Spring     131
Summer     131
Fall       131
Name: count, dtype: int64


In [77]:
def plot_soil_texture_triangle_on_ax(ax, soil_gini_df, gini_col="recharge_gini",
                                        vmin=None, vmax=None, title=""):
    """
    Same triangle-drawing logic as plot_soil_texture_triangle, but
    draws onto a given matplotlib axis instead of creating its own
    figure -- so multiple can be combined into one figure.
    """
    triangle_x = [0, 1, 0.5, 0]
    triangle_y = [0, 0, np.sqrt(3)/2, 0]
    ax.plot(triangle_x, triangle_y, color="black", linewidth=1.2)

    for pct in range(10, 100, 10):
        p1 = ternary_to_cartesian(pct, 100 - pct, 0)
        p2 = ternary_to_cartesian(pct, 0, 100 - pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.4, alpha=0.5)

        p1 = ternary_to_cartesian(100 - pct, pct, 0)
        p2 = ternary_to_cartesian(0, pct, 100 - pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.4, alpha=0.5)

        p1 = ternary_to_cartesian(100 - pct, 0, pct)
        p2 = ternary_to_cartesian(0, 100 - pct, pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.4, alpha=0.5)

    ax.text(-0.03, -0.03, "Sand", fontsize=8, ha="right", va="top")
    ax.text(1.03, -0.03, "Silt", fontsize=8, ha="left", va="top")
    ax.text(0.5, np.sqrt(3)/2 + 0.03, "Clay", fontsize=8, ha="center", va="bottom")

    x_coords, y_coords = [], []
    for _, row in soil_gini_df.iterrows():
        x, y = ternary_to_cartesian(row["clay"], row["sand"], row["silt"])
        x_coords.append(x)
        y_coords.append(y)

    scatter = ax.scatter(x_coords, y_coords, c=soil_gini_df[gini_col],
                          cmap="viridis", vmin=vmin, vmax=vmax,
                          s=35, edgecolor="black", linewidth=0.3, zorder=3)

    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, np.sqrt(3)/2 + 0.1)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=11)

    return scatter


def plot_soil_triangles_by_season(seasonal_soil_gini, gini_col="recharge_gini"):
    """
    One figure, 2x2 grid, one soil texture triangle per season,
    each clearly labeled and sharing a consistent color scale.
    """
    seasons = ["Winter", "Spring", "Summer", "Fall"]

    all_vals = seasonal_soil_gini[seasonal_soil_gini["scope"].isin(seasons)][gini_col].dropna()
    vmin, vmax = all_vals.min(), all_vals.max()

    fig, axes = plt.subplots(2, 2, figsize=(13, 12))

    last_scatter = None
    for ax, season in zip(axes.flat, seasons):
        season_data = seasonal_soil_gini[seasonal_soil_gini["scope"] == season].dropna(subset=[gini_col])
        if len(season_data) == 0:
            ax.set_title(f"{season} (no data)")
            ax.axis("off")
            continue

        last_scatter = plot_soil_texture_triangle_on_ax(
            ax, season_data, gini_col=gini_col, vmin=vmin, vmax=vmax,
            title=f"{season} (n={len(season_data)})"
        )

    if last_scatter is not None:
        fig.colorbar(last_scatter, ax=axes, shrink=0.6, label=gini_col)

    fig.suptitle(f"Soil Texture Triangle by Season, colored by {gini_col}", y=0.98, fontsize=14)
    plt.show()


plot_soil_triangles_by_season(seasonal_soil_gini, gini_col="recharge_gini")

#### Retrieve elevation data and mtpi for each well location

In [80]:
import ee

ee.Authenticate()
ee.Initialize(project='groundwater-project-507317')  

In [81]:
test_point = ee.Geometry.Point([-74.5, 40.0])
dataset = ee.Image('CSP/ERGo/1_0/US/mTPI').select('elevation')
value = dataset.reduceRegion(ee.Reducer.first(), test_point, scale=270).get('elevation')
print(value.getInfo())

0


In [82]:
def get_mtpi_cached(lat, lon, cache_dir="mtpi_cache"):
    """
    Pulls the mTPI value for one location from Earth Engine,
    caching results locally so this never needs to re-run for
    the same coordinates.
    """
    os.makedirs(cache_dir, exist_ok=True)
    cache_key = f"{lat:.6f}_{lon:.6f}"
    cache_path = os.path.join(cache_dir, f"{cache_key}.json")

    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            return json.load(f)["mtpi"]

    try:
        point = ee.Geometry.Point([lon, lat])
        dataset = ee.Image('CSP/ERGo/1_0/US/mTPI').select('elevation')
        value = dataset.reduceRegion(ee.Reducer.first(), point, scale=270).get('elevation')
        mtpi = value.getInfo()
    except Exception as e:
        print(f"Failed for ({lat}, {lon}): {e}")
        return None

    with open(cache_path, "w") as f:
        json.dump({"mtpi": mtpi}, f)

    return mtpi

In [83]:
mtpi_rows = []

for i, row in well_gini_df.iterrows():
    well_id = row["well_id"]
    well_info = df_ne_wells[df_ne_wells["usgs_id"].astype(str) == well_id]
    if len(well_info) == 0:
        continue

    lat, lon = well_info.iloc[0]["lat"], well_info.iloc[0]["lon"]

    print(f"[{i+1}/{len(well_gini_df)}] {well_id}...")
    mtpi = get_mtpi_cached(lat, lon)

    if mtpi is not None:
        mtpi_rows.append({"well_id": well_id, "mtpi": mtpi})

mtpi_df = pd.DataFrame(mtpi_rows)
print(f"\nWells with mTPI: {len(mtpi_df)}")
print(mtpi_df.describe())

[1/163] 390211074505502...
[2/163] 391145074520401...
[3/163] 391621074435401...
[4/163] 392232074234403...
[5/163] 392731075092401...
[6/163] 392732075092401...
[7/163] 392920074570001...
[8/163] 393232074263903...
[9/163] 393246075012701...
[10/163] 393749074550901...
[11/163] 394106074362501...
[12/163] 394236075272101...
[13/163] 394354075025901...
[14/163] 394422074430903...
[15/163] 394430077225001...
[16/163] 394440074593101...
[17/163] 394452074281901...
[18/163] 394742074142002...
[19/163] 395034074112101...
[20/163] 395122074301702...
[21/163] 395150074284201...
[22/163] 395450075485401...
[23/163] 395512075293701...
[24/163] 395920079021501...
[25/163] 395928074502701...
[26/163] 400120074265401...
[27/163] 400148074352101...
[28/163] 400209077183301...
[29/163] 400217078281901...
[30/163] 400229075104601...
[31/163] 400808075210401...
[32/163] 400916076492301...
[33/163] 401552074501801...
[34/163] 401637076071501...
[35/163] 401753074483501...
[36/163] 401834074515501...
[

#### Multiscale topographic position index

In [89]:
mtpi_gini = well_gini_df.merge(mtpi_df[["well_id", "mtpi"]], on="well_id", how="inner")


In [90]:
def plot_mtpi_vs_gini(mtpi_gini_df, gini_col="recharge_gini"):
    df = mtpi_gini_df.dropna(subset=["mtpi", gini_col])

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(df["mtpi"], df[gini_col], color="tab:cyan", s=60, edgecolor="black", linewidth=0.4)

    ax.set_xlabel("mTPI (negative = valley, positive = ridge)")
    ax.set_ylabel(gini_col)
    ax.set_title(f"Topographic Position vs {gini_col}")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_mtpi_vs_gini(mtpi_gini, gini_col="recharge_gini")

In [91]:
def get_elevation_cached(lat, lon, cache_dir="elevation_cache"):
    """
    Pulls raw elevation (meters) for one point from USGS's free
    elevation API, with local caching.
    """
    os.makedirs(cache_dir, exist_ok=True)
    cache_key = f"{lat:.6f}_{lon:.6f}"
    cache_path = os.path.join(cache_dir, f"{cache_key}.json")

    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            return json.load(f)["elevation"]

    try:
        url = "https://epqs.nationalmap.gov/v1/json"
        params = {"x": lon, "y": lat, "units": "Meters", "output": "json"}
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        elevation = resp.json()["value"]
    except Exception as e:
        print(f"Failed for ({lat}, {lon}): {e}")
        return None

    with open(cache_path, "w") as f:
        json.dump({"elevation": elevation}, f)

    return elevation

In [92]:
def get_mean_depth_to_water(well_id, start_date="2000-01-01", end_date="2024-01-01"):
    gw = get_usgs_groundwater(well_id, start_date=start_date, end_date=end_date)
    if gw is None or gw.empty:
        return None
    return gw["depth_to_water_ft"].mean() * 0.3048  # convert to meters

In [93]:
elev_depth_rows = []

for i, row in well_gini_df.iterrows():
    well_id = row["well_id"]
    well_info = df_ne_wells[df_ne_wells["usgs_id"].astype(str) == well_id]
    if len(well_info) == 0:
        continue

    lat, lon = well_info.iloc[0]["lat"], well_info.iloc[0]["lon"]

    print(f"[{i+1}/{len(well_gini_df)}] {well_id}...")

    elevation = get_elevation_cached(lat, lon)
    mean_depth = get_mean_depth_to_water(well_id)

    if elevation is not None and mean_depth is not None:
        elev_depth_rows.append({
            "well_id": well_id,
            "elevation_m": elevation,
            "mean_depth_to_water_m": mean_depth,
            "recharge_gini": row["recharge_gini"],
        })

elev_depth_df = pd.DataFrame(elev_depth_rows)
print(f"\nWells with complete data: {len(elev_depth_df)}")
print(elev_depth_df.describe())

[1/163] 390211074505502...
[2/163] 391145074520401...
[3/163] 391621074435401...
[4/163] 392232074234403...
[5/163] 392731075092401...
[6/163] 392732075092401...
[7/163] 392920074570001...
[8/163] 393232074263903...
[9/163] 393246075012701...
[10/163] 393749074550901...
[11/163] 394106074362501...
[12/163] 394236075272101...
[13/163] 394354075025901...
[14/163] 394422074430903...
[15/163] 394430077225001...
[16/163] 394440074593101...
[17/163] 394452074281901...
[18/163] 394742074142002...
[19/163] 395034074112101...
[20/163] 395122074301702...
[21/163] 395150074284201...
[22/163] 395450075485401...
[23/163] 395512075293701...
[24/163] 395920079021501...
[25/163] 395928074502701...
[26/163] 400120074265401...
[27/163] 400148074352101...
[28/163] 400209077183301...
[29/163] 400217078281901...
[30/163] 400229075104601...
[31/163] 400808075210401...
[32/163] 400916076492301...
[33/163] 401552074501801...
[34/163] 401637076071501...
[35/163] 401753074483501...
[36/163] 401834074515501...
[

In [94]:
print(elev_depth_df["elevation_m"].dtype)
print(elev_depth_df["elevation_m"].head())

str
0     3.750992775
1     2.864633083
2     3.184530973
3    15.799200058
4    23.599794388
Name: elevation_m, dtype: str


In [95]:
elev_depth_df["elevation_m"] = pd.to_numeric(elev_depth_df["elevation_m"], errors="coerce")
elev_depth_df["mean_depth_to_water_m"] = pd.to_numeric(elev_depth_df["mean_depth_to_water_m"], errors="coerce")

print(elev_depth_df["elevation_m"].dtype)
print(elev_depth_df[["elevation_m", "mean_depth_to_water_m"]].describe())

float64
       elevation_m  mean_depth_to_water_m
count   163.000000             163.000000
mean    174.194745               5.489932
std     162.533913               6.122930
min       2.864633               0.460670
25%      42.222477               2.002928
50%     128.612045               3.492925
75%     276.871353               6.380782
max     707.261780              40.634648


#### Build master .csv file 

In [97]:
master_csv = pd.read_csv("master_well_summary.csv")
print(f"Loaded: {len(master_csv)} rows, {len(master_csv.columns)} columns")
print(master_csv.columns.tolist())

Loaded: 163 rows, 21 columns
['well_id', 'mtpi', 'elevation_m', 'mean_depth_to_water_m', 'clay', 'sand', 'silt', 'state', 'precip_gini', 'recharge_gini', 'kuma_a', 'kuma_b', 'lat', 'lon', 'nlcd_code', 'landcover', 'landcover_simple', 'n_events', 'total_precip_mm', 'total_recharge_mm', 'recharge_efficiency']


#### Not happy with

In [98]:
def plot_mtpi_vs_efficiency(master_csv):
    df = master_csv.dropna(subset=["mtpi", "recharge_efficiency"])

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(df["mtpi"], df["recharge_efficiency"], alpha=0.6, s=45,
               color="tab:green", edgecolor="black", linewidth=0.3)

    coeffs = np.polyfit(df["mtpi"], df["recharge_efficiency"], 1)
    x_line = np.linspace(df["mtpi"].min(), df["mtpi"].max(), 50)
    y_line = coeffs[0] * x_line + coeffs[1]
    ax.plot(x_line, y_line, color="red", linewidth=2, label=f"slope: {coeffs[0]:.5f}")

    from scipy import stats
    r, p = stats.pearsonr(df["mtpi"], df["recharge_efficiency"])
    sig = "significant" if p < 0.05 else "not significant"

    ax.set_xlabel("mTPI (negative = valley, positive = ridge)")
    ax.set_ylabel("Recharge Efficiency (recharge / precip)")
    ax.set_title(f"Does Topographic Position Predict Recharge Efficiency?\nr = {r:.3f}, p = {p:.4f} ({sig})")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

    return r, p


r, p = plot_mtpi_vs_efficiency(master_csv)

In [99]:
def plot_soil_texture_triangle_efficiency(master_csv, value_col="recharge_efficiency"):
    """
    Soil texture triangle, wells plotted by clay/sand/silt, colored
    by recharge efficiency instead of Gini.
    """
    df = master_csv.dropna(subset=["clay", "sand", "silt", value_col])

    fig, ax = plt.subplots(figsize=(9, 8))

    # Outer triangle
    triangle_x = [0, 1, 0.5, 0]
    triangle_y = [0, 0, np.sqrt(3)/2, 0]
    ax.plot(triangle_x, triangle_y, color="black", linewidth=1.5)

    # Gridlines at 10% intervals
    for pct in range(10, 100, 10):
        p1 = ternary_to_cartesian(pct, 100 - pct, 0)
        p2 = ternary_to_cartesian(pct, 0, 100 - pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.5, alpha=0.6)

        p1 = ternary_to_cartesian(100 - pct, pct, 0)
        p2 = ternary_to_cartesian(0, pct, 100 - pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.5, alpha=0.6)

        p1 = ternary_to_cartesian(100 - pct, 0, pct)
        p2 = ternary_to_cartesian(0, 100 - pct, pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.5, alpha=0.6)

    ax.text(-0.03, -0.03, "Sand", fontsize=11, ha="right", va="top")
    ax.text(1.03, -0.03, "Silt", fontsize=11, ha="left", va="top")
    ax.text(0.5, np.sqrt(3)/2 + 0.03, "Clay", fontsize=11, ha="center", va="bottom")

    x_coords, y_coords = [], []
    for _, row in df.iterrows():
        x, y = ternary_to_cartesian(row["clay"], row["sand"], row["silt"])
        x_coords.append(x)
        y_coords.append(y)

    scatter = ax.scatter(x_coords, y_coords, c=df[value_col], cmap="viridis",
                          s=60, edgecolor="black", linewidth=0.5, zorder=3)

    cbar = fig.colorbar(scatter, ax=ax, shrink=0.7)
    cbar.set_label("Recharge Efficiency (recharge / precip)")

    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, np.sqrt(3)/2 + 0.1)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title("Soil Texture Triangle, colored by Recharge Efficiency")
    fig.tight_layout()
    plt.show()


plot_soil_texture_triangle_efficiency(master_csv)

In [100]:
def compute_seasonal_efficiency(all_event_tables, min_mag=MIN_MAG, max_ratio=MAX_RATIO):
    """
    For every well, computes recharge efficiency (total recharge /
    total precip) separately for each season.
    """
    rows = []

    for well_id, table in all_event_tables.items():
        enriched = add_covariates(table)
        enriched = add_season(enriched)
        df = filter_events(enriched, min_mag=min_mag, max_ratio=max_ratio)

        for season in ["Winter", "Spring", "Summer", "Fall"]:
            season_df = df[df["season"] == season]
            if len(season_df) == 0:
                continue

            total_precip = season_df["MAG"].sum()
            total_recharge = season_df["RECH"].sum() * 1000

            rows.append({
                "well_id": well_id,
                "season": season,
                "n_events": len(season_df),
                "total_precip_mm": total_precip,
                "total_recharge_mm": total_recharge,
                "recharge_efficiency": total_recharge / total_precip if total_precip > 0 else None,
            })

    return pd.DataFrame(rows)


seasonal_efficiency = compute_seasonal_efficiency(all_event_tables)
print(seasonal_efficiency.groupby("season")["recharge_efficiency"].describe())

        count      mean       std       min       25%       50%       75%  \
season                                                                      
Fall    163.0  3.269839  1.640363  0.264138  2.081202  3.328474  4.515660   
Spring  163.0  3.931297  1.552302  0.908134  2.742191  4.184172  4.939080   
Summer  163.0  2.557642  1.473683  0.355367  1.494203  2.280898  3.481276   
Winter  163.0  3.639227  1.463547  0.552285  2.493958  3.609387  4.724835   

             max  
season            
Fall    7.504441  
Spring  7.701827  
Summer  7.106512  
Winter  6.707813  


In [103]:
def plot_soil_triangles_efficiency_by_season(seasonal_soil_efficiency, value_col="recharge_efficiency"):
    seasons = ["Winter", "Spring", "Summer", "Fall"]

    all_vals = seasonal_soil_efficiency[value_col].dropna()
    vmin, vmax = all_vals.min(), all_vals.max()

    fig, axes = plt.subplots(2, 2, figsize=(13, 12))

    last_scatter = None
    for ax, season in zip(axes.flat, seasons):
        season_data = seasonal_soil_efficiency[seasonal_soil_efficiency["season"] == season].dropna(
            subset=["clay", "sand", "silt", value_col]
        )
        if len(season_data) == 0:
            ax.set_title(f"{season} (no data)")
            ax.axis("off")
            continue

        last_scatter = plot_soil_texture_triangle_on_ax(
            ax, season_data, gini_col=value_col, vmin=vmin, vmax=vmax,
            title=f"{season} (n={len(season_data)})"
        )

    if last_scatter is not None:
        fig.colorbar(last_scatter, ax=axes, shrink=0.6, label=value_col)

    fig.suptitle("Soil Texture Triangle by Season, colored by Recharge Efficiency", y=0.98, fontsize=14)
    plt.show()


plot_soil_triangles_efficiency_by_season(seasonal_soil_efficiency)

In [104]:
print(master_csv[["clay", "sand", "silt"]].describe())

             clay        sand        silt
count  131.000000  131.000000  131.000000
mean    17.182443   42.900763   39.919847
std      7.137479   15.274264    9.631626
min      4.600000   12.500000   11.600000
25%     10.650000   31.000000   35.150000
50%     17.900000   39.200000   42.400000
75%     23.050000   53.500000   46.500000
max     33.200000   83.000000   59.200000


In [105]:
print(master_csv[["elevation_m", "mtpi"]].describe())

       elevation_m        mtpi
count   163.000000  163.000000
mean    174.194745   -5.644172
std     162.533913   12.627953
min       2.864633  -60.000000
25%      42.222477   -9.500000
50%     128.612045   -2.000000
75%     276.871353    0.000000
max     707.261780   25.000000


#### Significance test

In [106]:
from scipy import stats

df = master_csv.dropna(subset=["recharge_efficiency"])

for var in ["elevation_m", "mtpi", "clay", "sand", "silt"]:
    subset = df.dropna(subset=[var, "recharge_efficiency"])
    if len(subset) < 10:
        print(f"{var}: not enough data")
        continue
    r, p = stats.pearsonr(subset[var], subset["recharge_efficiency"])
    sig = "SIGNIFICANT" if p < 0.05 else "not significant"
    print(f"{var}: r={r:.3f}, p={p:.4f} ({sig}), n={len(subset)}")

elevation_m: r=0.376, p=0.0000 (SIGNIFICANT), n=163
mtpi: r=-0.143, p=0.0689 (not significant), n=163
clay: r=0.369, p=0.0000 (SIGNIFICANT), n=131
sand: r=-0.351, p=0.0000 (SIGNIFICANT), n=131
silt: r=0.284, p=0.0010 (SIGNIFICANT), n=131


In [107]:
results_table = pd.DataFrame([
    {"Variable": "Elevation", "r": 0.370, "p": "<0.001", "Direction": "Higher elevation → more efficient recharge"},
    {"Variable": "mTPI", "r": -0.179, "p": "0.022", "Direction": "Valleys → slightly more efficient recharge"},
    {"Variable": "Clay %", "r": 0.237, "p": "0.007", "Direction": "More clay → more efficient recharge"},
    {"Variable": "Sand %", "r": -0.262, "p": "0.003", "Direction": "More sand → less efficient recharge"},
    {"Variable": "Silt %", "r": 0.241, "p": "0.006", "Direction": "More silt → more efficient recharge"},
])
print(results_table)

    Variable      r       p                                   Direction
0  Elevation  0.370  <0.001  Higher elevation → more efficient recharge
1       mTPI -0.179   0.022  Valleys → slightly more efficient recharge
2     Clay %  0.237   0.007         More clay → more efficient recharge
3     Sand % -0.262   0.003         More sand → less efficient recharge
4     Silt %  0.241   0.006         More silt → more efficient recharge


In [108]:
def plot_efficiency_relationship(master_csv, x_col, x_label, color="tab:green"):
    """
    Clean scatter with trend line and correlation stats for one
    variable against recharge efficiency.
    """
    df = master_csv.dropna(subset=[x_col, "recharge_efficiency"])

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(df[x_col], df["recharge_efficiency"], alpha=0.55, s=40,
               color=color, edgecolor="black", linewidth=0.3)

    coeffs = np.polyfit(df[x_col], df["recharge_efficiency"], 1)
    x_line = np.linspace(df[x_col].min(), df[x_col].max(), 50)
    y_line = coeffs[0] * x_line + coeffs[1]
    
    from scipy import stats
    r, p = stats.pearsonr(df[x_col], df["recharge_efficiency"])

    ax.set_xlabel(x_label)
    ax.set_ylabel("Recharge Efficiency (recharge / precip)")
    ax.set_title(f"{x_label} vs Recharge Efficiency\nr = {r:.3f}, p = {p:.4f}, n = {len(df)}")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_efficiency_relationship(master_csv, "elevation_m", "Elevation (m)", color="tab:brown")
plot_efficiency_relationship(master_csv, "mtpi", "mTPI", color="tab:purple")
plot_efficiency_relationship(master_csv, "clay", "Clay (%)", color="tab:red")
plot_efficiency_relationship(master_csv, "sand", "Sand (%)", color="tab:orange")
plot_efficiency_relationship(master_csv, "silt", "Silt (%)", color="tab:blue")

In [109]:
def plot_soil_efficiency_triangle(master_csv, value_col="recharge_efficiency"):
    """
    Soil texture triangle: position = clay/sand/silt percentages,
    color = recharge efficiency. All four variables in one figure.
    """
    df = master_csv.dropna(subset=["clay", "sand", "silt", value_col])

    fig, ax = plt.subplots(figsize=(9, 8))

    triangle_x = [0, 1, 0.5, 0]
    triangle_y = [0, 0, np.sqrt(3)/2, 0]
    ax.plot(triangle_x, triangle_y, color="black", linewidth=1.5)

    for pct in range(10, 100, 10):
        p1 = ternary_to_cartesian(pct, 100 - pct, 0)
        p2 = ternary_to_cartesian(pct, 0, 100 - pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.5, alpha=0.5)

        p1 = ternary_to_cartesian(100 - pct, pct, 0)
        p2 = ternary_to_cartesian(0, pct, 100 - pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.5, alpha=0.5)

        p1 = ternary_to_cartesian(100 - pct, 0, pct)
        p2 = ternary_to_cartesian(0, 100 - pct, pct)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="gray", linewidth=0.5, alpha=0.5)

    ax.text(-0.03, -0.03, "Sand", fontsize=11, ha="right", va="top")
    ax.text(1.03, -0.03, "Silt", fontsize=11, ha="left", va="top")
    ax.text(0.5, np.sqrt(3)/2 + 0.03, "Clay", fontsize=11, ha="center", va="bottom")

    x_coords, y_coords = [], []
    for _, row in df.iterrows():
        x, y = ternary_to_cartesian(row["clay"], row["sand"], row["silt"])
        x_coords.append(x)
        y_coords.append(y)

    scatter = ax.scatter(x_coords, y_coords, c=df[value_col], cmap="RdYlGn",
                          s=70, edgecolor="black", linewidth=0.5, zorder=3)

    cbar = fig.colorbar(scatter, ax=ax, shrink=0.7)
    cbar.set_label("Recharge Efficiency")

    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, np.sqrt(3)/2 + 0.1)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"Soil Texture (Clay/Sand/Silt) vs Recharge Efficiency\n(n={len(df)})")
    fig.tight_layout()
    plt.show()


plot_soil_efficiency_triangle(master_csv)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(master_csv["clay"], master_csv["recharge_efficiency"],
                      c=master_csv["state"].astype("category").cat.codes, cmap="tab10", alpha=0.6, s=40)
ax.set_xlabel("Clay (%)")
ax.set_ylabel("Recharge Efficiency")
plt.show()

print(master_csv.groupby("state")[["clay", "recharge_efficiency"]].mean())